# moonshot_compounding — Test 1 (Momentum) — **PHASE A: Validation Notebook (v3)**

**Run against the v3 panel** (`feature_panel_with_pit_v2_*` rebuilt from
`raw_price_panel_consolidated_v3`). A0 hard-stops if pointed at a pre-fix panel.

**Status:** run once, read the output, archive. This notebook is the audit trail.
It does **not** run the test.

---

## What changed since the first Phase A run

The first run passed every correctness check — zero leakage, spot-checks exact to
1e-12, PIT filter demonstrably varying — but surfaced a **panel defect**: yfinance
fabricated rows on six NSE holidays, which Stage 3B masked but did not remove.
Each fabricated holiday killed two returns (its own, and the next session's, which
had no prior close) for **every ticker simultaneously**, so every stock's 232-day
window failed. The momentum-eligible universe collapsed from ~480 to **126 in 2025
and 22 in 2026**.

Those six dates are now deleted in v3. This notebook re-measures against the fix.

**One date was deliberately NOT deleted.** `2025-03-18` looked identical — 98% of
rows masked — but NSE's published circular says the market was open, and the 19
surviving rows show real trading (ITC 13.9M shares, SpiceJet 13.3M). That is a
**vendor failure**, not a closure: a live session where yfinance failed to deliver.
Deleting it would have spliced returns across a day that actually happened.

| Change | Cell |
|---|---|
| Panel constants: 3,328,132 rows / 4,237 dates | A0 |
| **v3 provenance guard** — halts if any deleted date is present | A0 |
| `date_status` added to required columns | A0 |
| **Vendor-failure adjustment** to the strict filter | A2 |
| Vendor-failure impact reported explicitly | A2, A7 |
| Q4 added: did the fix restore the collapsed years? | A7 |

---

## The vendor-failure adjustment (the one substantive logic change)

A vendor outage is **not a property of the stock**. Requiring 232/232 valid returns
across a window spanning `2025-03-18` would reject nearly every name for something
the data vendor did — the identical failure mode the six fabricated holidays caused,
at smaller scale.

So the rule becomes what it always meant: every return in the window must be valid
**except** on vendor-failure dates, which drop out of the denominator. `n_required`
shrinks per-window, so a window that doesn't span one is unaffected.

The successor row (whose return is NaN because its prior close is missing) is derived
**structurally**, not hardcoded — so this stays correct if the tagging ever changes.

Excluded rows are dropped from the compounding product rather than treated as zero
returns. A zero would be a fabricated value, and this project never substitutes a
plausible number for an unknown one.

---

## Confirmed specification

| Decision | Value |
|---|---|
| Rebalance | Last **trading** day of each month, from the panel's own dates |
| Signal | `prod(1 + daily_return) - 1` over rows `[t-252, t-21]` = **232 rows** |
| Coverage filter | **Hard 232/232, less vendor-failure rows.** 186 (80%) as diagnostic only |
| PIT filter | `in_nifty500_flag == 1` — **never `!= 1`** (fact #17, tri-state) |
| Liquidity terciles | Trailing mean `turnover` over `[t-21, t-1]` — rebalance day **excluded** |
| Exit treatment | Settled: 37 true terminations, evenly spread. Compound observed days |

---

## Two things on record before any number is read

**1. The 232 filter changes what the test measures.** The result is *"momentum among
Nifty 500 names with a clean trailing year"*, not *"momentum in the Nifty 500"*. That
belongs in the log's `data_and_universe` field, not a footnote.

**2. This is harness validation.** Momentum in Indian equities is well documented. A
strong positive result is *expected* and is weak evidence the pipeline works. A **null
result is the informative outcome** — it would more likely indicate a pipeline bug
than a genuine absence of momentum.

---

## Known limitation carried forward

`realized_vol_120` is **unusable for yfinance-sourced names between roughly
2025-03-18 and 2025-09-30** — measured coverage drops from 93.2% to 1.5% and recovers
only as the 120-day window rolls past the vendor-failure date. This does not affect
momentum (which uses only `daily_return`), but it is a hard constraint on the
low-volatility hypothesis.

---

## Run instructions

Run top to bottom. A0 hard-stops on any preflight failure. Runtime is dominated by
A2 and A5; expect roughly 15–30 minutes.

Only file writes are two artifacts in `validation/`. **Nothing in the panel is mutated.**

## Cell A0 — Environment, resolver, preflight

Mounts Drive, detects the stale-mount pathology (fact #23), resolves the panel
newest-wins, and runs hard asserts on schema, row count, date range, and the
PIT flag tri-state invariant **before** any analysis touches the data.

Hard-stops if anything fails. A shape or schema mismatch means this notebook's
assumptions no longer describe the panel — that is a genuine blocker, not a warning.

In [1]:
# =============================================================================
# CELL A0 -- Environment, resolver, preflight
# =============================================================================
# WHAT THIS CELL DOES:
#   1. Mounts Google Drive and verifies the mount is NOT stale (fact #23).
#   2. Defines the resolve() helper -- newest-wins file resolution.
#   3. Loads the Stage 6 deliverable feature panel.
#   4. Runs hard asserts on schema, row count, date range, and PIT flag
#      tri-state invariant BEFORE any analysis touches the data.
#
# WHY IT MATTERS:
#   A stale Colab Drive mount presents as an "empty project folder" -- the
#   directory tree syncs but file CONTENTS do not. This is not lost data, but
#   it will silently produce garbage if not caught. We fail loudly instead.
#
# NOTHING IS WRITTEN IN THIS CELL. Read-only.
# =============================================================================

import os
import sys
import glob
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

# Show all columns when printing diagnostic tables -- we want to eyeball things.
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

# -----------------------------------------------------------------------------
# RUN_LOG: every check appends (name, status, detail) here. Cell A7 replays it.
# We use a plain list of tuples rather than prints alone so that the final
# summary cell can RE-READ results rather than RE-COMPUTE them.
# -----------------------------------------------------------------------------
RUN_LOG = []
FAILED_CHECKS = []


def check(name, condition, detail=""):
    """Record a pass/fail check. Never raises -- we want the full picture of
    what passed and what failed, not an abort on the first problem. Cell A0
    ends with an explicit hard-stop if anything critical failed."""
    status = "PASS" if bool(condition) else "FAIL"
    RUN_LOG.append((name, status, str(detail)))
    if status == "FAIL":
        FAILED_CHECKS.append((name, detail))
    print(f"  [{status}] {name}" + (f"  --  {detail}" if detail else ""))
    return bool(condition)


# -----------------------------------------------------------------------------
# Drive mount
# -----------------------------------------------------------------------------
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("NOTE: not running in Colab. PROJECT_ROOT must be set manually below.")

PROJECT_ROOT = "/content/drive/MyDrive/moonshot_compounding"

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"\nRun timestamp: {RUN_TIMESTAMP}")
print(f"Project root:  {PROJECT_ROOT}")

# -----------------------------------------------------------------------------
# STALE MOUNT PREFLIGHT
# A stale mount shows directories but zero readable parquet files.
# -----------------------------------------------------------------------------
print("\n--- Preflight: mount health ---")

_root_exists = os.path.isdir(PROJECT_ROOT)
check("project root directory exists", _root_exists, PROJECT_ROOT)

if not _root_exists:
    raise RuntimeError(
        f"Project root not found at {PROJECT_ROOT}.\n"
        "If the folder LOOKS present in the Drive UI, this is the stale-mount\n"
        "pathology (fact #23). Fix: Runtime > Disconnect and delete runtime,\n"
        "then re-run this notebook from the top."
    )

_all_parquet = glob.glob(os.path.join(PROJECT_ROOT, "**", "*.parquet"), recursive=True)
check(
    "parquet files visible under project root",
    len(_all_parquet) > 0,
    f"{len(_all_parquet)} files found",
)

if len(_all_parquet) == 0:
    raise RuntimeError(
        "Project tree is visible but contains ZERO parquet files.\n"
        "This is the stale-mount pathology (fact #23) -- the directory tree\n"
        "syncs but file contents do not. Your data is NOT lost.\n"
        "Fix: Runtime > Disconnect and delete runtime, then re-run."
    )


# -----------------------------------------------------------------------------
# resolve() -- newest-wins file resolution.
# Reads are date-independent by design: we never hard-code a date suffix,
# because the panel filename carries a build date that will change.
# -----------------------------------------------------------------------------
def resolve(relative_glob, must_exist=True):
    """Return the newest file matching a glob pattern relative to PROJECT_ROOT.

    'Newest' is by filesystem modification time, NOT by lexical filename sort,
    because a v2 file built on an earlier date should still lose to a v2 file
    built later. Prints the resolution so every run is traceable.
    """
    pattern = os.path.join(PROJECT_ROOT, relative_glob)
    matches = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
    if not matches:
        if must_exist:
            raise FileNotFoundError(f"No file matched pattern: {pattern}")
        return None
    if len(matches) > 1:
        print(f"  NOTE: {len(matches)} files matched '{relative_glob}'. "
              f"Taking newest by mtime.")
        for m in matches[:5]:
            _mt = datetime.fromtimestamp(os.path.getmtime(m)).strftime("%Y-%m-%d %H:%M")
            print(f"        {'-> ' if m == matches[0] else '   '}{os.path.basename(m)}  (mtime {_mt})")
    return matches[0]


# -----------------------------------------------------------------------------
# Load the panel
# -----------------------------------------------------------------------------
print("\n--- Resolving feature panel ---")

PANEL_PATH = resolve("features/feature_panel_with_pit_v2_*.parquet")
PANEL_FILENAME = os.path.basename(PANEL_PATH)
PANEL_MTIME = datetime.fromtimestamp(os.path.getmtime(PANEL_PATH)).strftime("%Y-%m-%d %H:%M:%S")

print(f"  Resolved: {PANEL_FILENAME}")
print(f"  Modified: {PANEL_MTIME}")
print(f"  Size:     {os.path.getsize(PANEL_PATH) / 1e6:,.1f} MB")

print("\n--- Loading panel (this takes ~30-60s) ---")
panel = pd.read_parquet(PANEL_PATH)
print(f"  Loaded. Shape: {panel.shape[0]:,} rows x {panel.shape[1]} columns")

print("\n--- Columns present in panel ---")
for c in panel.columns:
    print(f"    {c:<40} {str(panel[c].dtype)}")


# -----------------------------------------------------------------------------
# Identify the ticker and date key columns.
# The memory doc does not record their exact names, so we detect rather than
# assume -- and we FAIL LOUDLY if detection is ambiguous rather than guessing.
# -----------------------------------------------------------------------------
print("\n--- Preflight: key column detection ---")

_ticker_candidates = [c for c in panel.columns
                      if c.lower() in ("ticker", "symbol", "tickers", "sym", "yf_ticker")]
_date_candidates = [c for c in panel.columns
                    if c.lower() in ("date", "dt", "trade_date", "datetime")]

check("exactly one ticker key column detected", len(_ticker_candidates) == 1,
      f"candidates={_ticker_candidates}")
check("exactly one date key column detected", len(_date_candidates) == 1,
      f"candidates={_date_candidates}")

if len(_ticker_candidates) != 1 or len(_date_candidates) != 1:
    raise RuntimeError(
        "Could not unambiguously identify the ticker/date key columns.\n"
        f"Ticker candidates: {_ticker_candidates}\n"
        f"Date candidates:   {_date_candidates}\n"
        "Set TICKER_COL and DATE_COL manually and re-run. "
        "Do NOT guess -- a wrong key silently corrupts every groupby below."
    )

TICKER_COL = _ticker_candidates[0]
DATE_COL = _date_candidates[0]
print(f"  TICKER_COL = '{TICKER_COL}'")
print(f"  DATE_COL   = '{DATE_COL}'")

# Normalise the date column to tz-naive datetime64.
# yfinance returns IST tz-aware timestamps in some paths; the panel should
# already be naive, but we enforce it rather than trust it.
panel[DATE_COL] = pd.to_datetime(panel[DATE_COL])
if getattr(panel[DATE_COL].dtype, "tz", None) is not None:
    print("  NOTE: date column was tz-aware. Stripping tz to naive.")
    panel[DATE_COL] = panel[DATE_COL].dt.tz_localize(None)


# -----------------------------------------------------------------------------
# Required-column asserts
# -----------------------------------------------------------------------------
print("\n--- Preflight: required columns ---")

REQUIRED_COLS = ["daily_return", "in_nifty500_flag", "turnover", "price_status", "date_status"]
for col in REQUIRED_COLS:
    check(f"column present: {col}", col in panel.columns)

_missing = [c for c in REQUIRED_COLS if c not in panel.columns]
if _missing:
    raise RuntimeError(
        f"Required columns missing from panel: {_missing}\n"
        f"Panel columns are: {list(panel.columns)}\n"
        "NOTE: 'date_status' only exists on the v3 panel. If it is missing, this "
        "notebook is pointed at a PRE-FIX panel (v2 or earlier). Do NOT proceed "
        "-- rebuild v3 first, then re-run."
    )


# -----------------------------------------------------------------------------
# Shape / range asserts against the memory document.
# A mismatch here means the panel and the memory doc have DIVERGED, which is a
# genuine blocker, not a warning.
# -----------------------------------------------------------------------------
print("\n--- Preflight: panel shape and range vs IMPLEMENTER_MEMORY ---")

# v3 panel: v2 minus 4,747 rows on six market-closed dates (NSE holidays that
# yfinance fabricated rows for). See the Stage 3B amendment in the pipeline.
EXPECTED_ROWS = 3_328_132
EXPECTED_DATES = 4_237
EXPECTED_START = pd.Timestamp("2010-01-04")
EXPECTED_END = pd.Timestamp("2026-07-10")

# The six deleted dates. Phase A asserts they are ABSENT -- if any is present,
# this notebook is pointed at a pre-fix panel and every coverage number below
# would be measuring the old defect.
MARKET_CLOSED_REMOVED = [pd.Timestamp(d) for d in
    ["2014-04-24", "2014-10-15", "2026-01-15",
     "2026-05-01", "2026-05-28", "2026-06-26"]]

actual_rows = len(panel)
actual_tickers = panel[TICKER_COL].nunique()
actual_start = panel[DATE_COL].min()
actual_end = panel[DATE_COL].max()

actual_dates = panel[DATE_COL].nunique()

check("row count matches v3 panel (3,328,132)",
      actual_rows == EXPECTED_ROWS,
      f"actual={actual_rows:,} expected={EXPECTED_ROWS:,}")
check("distinct date count is 4,237", actual_dates == EXPECTED_DATES,
      f"actual={actual_dates:,}")
check("ticker count is 1,069", actual_tickers == 1069, f"actual={actual_tickers:,}")
check("panel starts 2010-01-04", actual_start == EXPECTED_START, f"actual={actual_start.date()}")
check("panel ends 2026-07-10", actual_end == EXPECTED_END, f"actual={actual_end.date()}")

# -----------------------------------------------------------------------------
# v3 PROVENANCE GUARD.
# The whole point of Phase A is measuring coverage. If it is accidentally run
# against a pre-fix panel, every number below would be measuring the defect we
# just removed -- and would look like a failed fix rather than a wrong input.
# -----------------------------------------------------------------------------
print("\n--- Preflight: v3 provenance (market-closed dates must be ABSENT) ---")
_present = [d for d in MARKET_CLOSED_REMOVED if (panel[DATE_COL] == d).any()]
check("none of the six market-closed dates present in panel",
      len(_present) == 0,
      f"present={[str(d.date()) for d in _present]}")
if _present:
    raise RuntimeError(
        f"Market-closed dates still present: {[str(d.date()) for d in _present]}\n"
        "This is a PRE-FIX panel. Rebuild v3 via the Stage 3B amendment and "
        "re-run the Stage 4-6 pipeline before running Phase A.")

_vf = panel[panel["date_status"] == "vendor_failure"]
_vf_dates = sorted(pd.to_datetime(_vf[DATE_COL].unique()))
print(f"  vendor_failure rows: {len(_vf):,} on {len(_vf_dates)} date(s): "
      f"{[str(pd.Timestamp(d).date()) for d in _vf_dates]}")
check("exactly one vendor_failure date (2025-03-18)",
      len(_vf_dates) == 1 and pd.Timestamp(_vf_dates[0]) == pd.Timestamp("2025-03-18"),
      f"{[str(pd.Timestamp(d).date()) for d in _vf_dates]}")
VENDOR_FAILURE_DATES = set(pd.Timestamp(d) for d in _vf_dates)

PANEL_ROWS = actual_rows
PANEL_DATES = actual_dates
PANEL_TICKERS = actual_tickers
PANEL_START = actual_start
PANEL_END = actual_end


# -----------------------------------------------------------------------------
# PIT flag tri-state invariant (fact #17).
# We re-verify this independently rather than trusting the Stage 4 assert.
# -----------------------------------------------------------------------------
print("\n--- Preflight: PIT flag tri-state invariant (fact #17) ---")

flag = panel["in_nifty500_flag"]
_nonnull_vals = set(pd.unique(flag.dropna()))
check("flag takes only values {0, 1} (plus NaN)",
      _nonnull_vals.issubset({0, 1, 0.0, 1.0, True, False}),
      f"observed non-null values={sorted(_nonnull_vals, key=str)}")

n_flag_nan = int(flag.isna().sum())
n_flag_1 = int((flag == 1).sum())
n_flag_0 = int((flag == 0).sum())
check("flag counts sum to panel rows",
      n_flag_nan + n_flag_1 + n_flag_0 == PANEL_ROWS,
      f"NaN={n_flag_nan:,} one={n_flag_1:,} zero={n_flag_0:,}")

CUTOVER = pd.Timestamp("2010-06-30")
_nan_after_cutover = int((flag.isna() & (panel[DATE_COL] >= CUTOVER)).sum())
check("NO NaN flag on/after 2010-06-30 (hard invariant)",
      _nan_after_cutover == 0,
      f"violations={_nan_after_cutover:,}")

_nan_before_cutover = int((flag.isna() & (panel[DATE_COL] < CUTOVER)).sum())
print(f"  INFO: {_nan_before_cutover:,} NaN flags before 2010-06-30 "
      f"(expected ~85,464 -- these are genuinely UNKNOWN, not known-out).")
RUN_LOG.append(("pre-cutover NaN flag count", "INFO", f"{_nan_before_cutover:,}"))


# -----------------------------------------------------------------------------
# Sort once, canonically. Every downstream positional operation depends on this.
# -----------------------------------------------------------------------------
print("\n--- Sorting panel canonically by (ticker, date) ---")
panel = panel.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)

_dupes = int(panel.duplicated(subset=[TICKER_COL, DATE_COL]).sum())
check("no duplicate (ticker, date) rows", _dupes == 0, f"duplicates={_dupes:,}")

# -----------------------------------------------------------------------------
# DATETIME UNIT CONSISTENCY.
# np.datetime64 hashes and compares by UNIT as well as by value. A datetime64[ns]
# panel column combined with a datetime64[us] lookup key produces a SILENT total
# lookup failure -- every signal comes back NaN with no error raised. This was a
# real 100%-failure bug caught on first live run. We normalise explicitly and
# assert, so it can never recur silently.
# -----------------------------------------------------------------------------
print("\n--- Preflight: datetime unit consistency ---")
_orig_unit = str(panel[DATE_COL].dtype)
panel[DATE_COL] = panel[DATE_COL].astype("datetime64[ns]")
check("date column normalised to datetime64[ns]",
      str(panel[DATE_COL].dtype) == "datetime64[ns]",
      f"was {_orig_unit}, now {panel[DATE_COL].dtype}")

_probe = panel[DATE_COL].to_numpy()[0]
_roundtrip = np.datetime64(pd.Timestamp(_probe), "ns")
check("np.datetime64(pd.Timestamp(x),'ns') round-trips to an equal value",
      _probe == _roundtrip, f"probe={_probe} roundtrip={_roundtrip}")


# -----------------------------------------------------------------------------
# Hard stop if anything critical failed.
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
if FAILED_CHECKS:
    print(f"CELL A0: {len(FAILED_CHECKS)} CHECK(S) FAILED")
    for nm, dt in FAILED_CHECKS:
        print(f"   - {nm}: {dt}")
    print("=" * 78)
    raise RuntimeError(
        "Preflight failed. Do NOT proceed to A1+. A shape/schema mismatch "
        "means this notebook's assumptions no longer describe the panel."
    )
else:
    print("CELL A0: ALL PREFLIGHT CHECKS PASSED")
    print("=" * 78)

Mounted at /content/drive

Run timestamp: 2026-07-24 14:08:17
Project root:  /content/drive/MyDrive/moonshot_compounding

--- Preflight: mount health ---
  [PASS] project root directory exists  --  /content/drive/MyDrive/moonshot_compounding
  [PASS] parquet files visible under project root  --  1592 files found

--- Resolving feature panel ---
  NOTE: 4 files matched 'features/feature_panel_with_pit_v2_*.parquet'. Taking newest by mtime.
        -> feature_panel_with_pit_v2_2026-07-24.parquet  (mtime 2026-07-24 07:52)
           feature_panel_with_pit_v2_2026-07-22.parquet  (mtime 2026-07-22 06:02)
           feature_panel_with_pit_v2_2026-07-21.parquet  (mtime 2026-07-21 10:39)
           feature_panel_with_pit_v2_2026-07-20.parquet  (mtime 2026-07-20 11:54)
  Resolved: feature_panel_with_pit_v2_2026-07-24.parquet
  Modified: 2026-07-24 07:52:10
  Size:     251.5 MB

--- Loading panel (this takes ~30-60s) ---
  Loaded. Shape: 3,328,132 rows x 23 columns

--- Columns present in panel 

## Cell A1 — Rebalance calendar

Last **trading** day of each month, taken as the max observed date within each
month across the whole panel. Deriving from the panel's own dates rather than a
synthetic calendar means the NSE holiday calendar is handled by construction —
no rebalance can land on Diwali.

Expect ~198 dates (Jan 2010 → Jul 2026). Eyeball the printed sample: any
"month-end" falling on a weekend means the calendar was built wrong.

In [2]:
# =============================================================================
# CELL A1 -- Rebalance calendar
# =============================================================================
# WHAT THIS CELL DOES:
#   Builds the list of rebalance dates: the LAST TRADING DAY of each calendar
#   month present in the panel.
#
# WHY WE DERIVE TRADING DAYS FROM THE PANEL ITSELF:
#   We do NOT use a synthetic calendar (e.g. pandas BMonthEnd), because that
#   would place rebalances on Indian market holidays -- Diwali, Holi, etc. By
#   taking the max observed date within each month across the whole panel, the
#   NSE holiday calendar is handled by construction.
#
# EXPECTED: ~198 month-ends (Jan 2010 -> Jul 2026). The FINAL rebalance has no
#   forward month to measure and will be dropped in Phase B, not here -- Phase A
#   is diagnostic only and we want to see it.
#
# NOTHING IS WRITTEN. Read-only.
# =============================================================================

print("--- Building rebalance calendar from panel's own trading days ---")

# All distinct trading dates observed anywhere in the panel.
# Using the union across tickers (not any single ticker) because no individual
# ticker is guaranteed to have traded on every market day.
all_trading_days = pd.Series(sorted(panel[DATE_COL].unique()))
all_trading_days = pd.to_datetime(all_trading_days)

print(f"  Distinct trading days in panel: {len(all_trading_days):,}")
print(f"  First: {all_trading_days.iloc[0].date()}   "
      f"Last: {all_trading_days.iloc[-1].date()}")

# Group trading days by (year, month) and take the max -> last trading day.
_td = pd.DataFrame({"date": all_trading_days})
_td["ym"] = _td["date"].dt.to_period("M")
rebalance_dates = (
    _td.groupby("ym", as_index=False)["date"]
       .max()
       .rename(columns={"date": "rebalance_date"})
       .sort_values("rebalance_date")
       .reset_index(drop=True)
)

REBALANCE_DATES = list(rebalance_dates["rebalance_date"])
N_REBALANCES = len(REBALANCE_DATES)

print(f"\n  Rebalance dates constructed: {N_REBALANCES}")
print(f"  First rebalance: {REBALANCE_DATES[0].date()}")
print(f"  Last  rebalance: {REBALANCE_DATES[-1].date()}")

check("rebalance count is in the expected 190-205 range",
      190 <= N_REBALANCES <= 205,
      f"actual={N_REBALANCES}")

# -----------------------------------------------------------------------------
# EYEBALL CHECK: are these genuinely month-ends, and genuinely weekdays?
# A "month-end" landing on a Saturday/Sunday means the calendar was built wrong.
# -----------------------------------------------------------------------------
print("\n--- Eyeball sample of rebalance dates (first 3, middle 3, last 3) ---")
_sample_idx = list(range(3)) + \
              list(range(N_REBALANCES // 2, N_REBALANCES // 2 + 3)) + \
              list(range(N_REBALANCES - 3, N_REBALANCES))

print(f"  {'rebalance_date':<16} {'weekday':<11} {'days_to_month_end'}")
for i in _sample_idx:
    d = REBALANCE_DATES[i]
    # Calendar month end for that month:
    cal_month_end = d + pd.offsets.MonthEnd(0)
    gap = (cal_month_end - d).days
    print(f"  {str(d.date()):<16} {d.strftime('%A'):<11} {gap}")

_weekend_rebalances = [d for d in REBALANCE_DATES if d.weekday() >= 5]
check("no rebalance date falls on a weekend",
      len(_weekend_rebalances) == 0,
      f"weekend dates={[str(d.date()) for d in _weekend_rebalances[:5]]}")

# A rebalance more than ~6 days before its calendar month end suggests either a
# long holiday cluster or a data gap. Worth surfacing, not necessarily an error.
_late_gaps = [(d, (d + pd.offsets.MonthEnd(0) - d).days)
              for d in REBALANCE_DATES
              if (d + pd.offsets.MonthEnd(0) - d).days > 6]
print(f"\n  Rebalances >6 days before calendar month-end: {len(_late_gaps)}")
for d, g in _late_gaps[:10]:
    print(f"    {d.date()}  ({g} days early -- check for holiday cluster/data gap)")
RUN_LOG.append(("rebalances >6d before month-end", "INFO", f"{len(_late_gaps)}"))

# -----------------------------------------------------------------------------
# Build a fast positional index of trading days.
# Used later to translate "252 trading rows back" into an actual date.
# -----------------------------------------------------------------------------
TRADING_DAY_INDEX = pd.Index(all_trading_days)
print(f"\n  TRADING_DAY_INDEX built: {len(TRADING_DAY_INDEX):,} entries")

print("\n" + "=" * 78)
print(f"CELL A1 COMPLETE: {N_REBALANCES} rebalance dates ready")
print("=" * 78)

--- Building rebalance calendar from panel's own trading days ---
  Distinct trading days in panel: 4,237
  First: 2010-01-04   Last: 2026-07-10

  Rebalance dates constructed: 199
  First rebalance: 2010-01-29
  Last  rebalance: 2026-07-10
  [PASS] rebalance count is in the expected 190-205 range  --  actual=199

--- Eyeball sample of rebalance dates (first 3, middle 3, last 3) ---
  rebalance_date   weekday     days_to_month_end
  2010-01-29       Friday      2
  2010-02-26       Friday      2
  2010-03-31       Wednesday   0
  2018-04-30       Monday      0
  2018-05-31       Thursday    0
  2018-06-29       Friday      1
  2026-05-29       Friday      2
  2026-06-30       Tuesday     0
  2026-07-10       Friday      21
  [FAIL] no rebalance date falls on a weekend  --  weekend dates=['2015-02-28']

  Rebalances >6 days before calendar month-end: 1
    2026-07-10  (21 days early -- check for holiday cluster/data gap)

  TRADING_DAY_INDEX built: 4,237 entries

CELL A1 COMPLETE: 199 r

## Cell A2 — Signal construction + coverage

`momentum_signal(t) = prod(1 + daily_return) - 1` over the **232 trading rows**
spanning `[t-252, t-21]` inclusive.

Three design decisions made visible in code rather than comments:

- **Offsets are positional within each ticker's own sorted history** — not
  calendar days, not positions in a global calendar. A ticker with gaps must not
  be silently shifted relative to one without.
- **`as_of` is an explicit required parameter** of `compute_momentum_signal()`.
  The point-in-time assumption is part of the interface.
- **`n_valid_returns` is retained**, so the 186 diagnostic is a *filter on a
  column*, not a second computation that could drift from the first.

Coverage is reported strict-vs-relaxed side by side, with NaN attribution
separating structural warm-up from data-quality exclusion — these are different
problems and should not be summed. Fact #22's prediction of thin 2014/2025
coverage is **measured, not assumed**.

In [3]:
# =============================================================================
# CELL A2 -- Momentum signal construction + coverage diagnostic
# =============================================================================
# SIGNAL DEFINITION (as agreed):
#   momentum_signal(t) = prod(1 + daily_return[i]) - 1
#                        over the 232 TRADING ROWS spanning [t-252, t-21]
#
#   i.e. a 12-month lookback (252 trading rows) skipping the most recent month
#   (21 trading rows) to avoid short-term reversal contamination.
#
# CRITICAL DESIGN DECISIONS, MADE VISIBLE IN CODE NOT COMMENTS:
#
#   1. OFFSETS ARE POSITIONAL WITHIN EACH TICKER'S OWN SORTED HISTORY.
#      Not calendar days, and not positions in a global calendar. A ticker with
#      gaps must not be silently shifted relative to a ticker without gaps.
#
#   2. THE WINDOW IS STRICTLY IN THE PAST. Row t itself is EXCLUDED, and so are
#      the 21 rows before it. The last row entering the window is t-21.
#      Cell A3 proves this empirically rather than asserting it here.
#
#   3. STRICT COVERAGE: all 232 returns must be valid (non-NaN) or the signal
#      is NaN. This is your decision. Note it changes what the test measures --
#      "momentum among names with a clean trailing year", not "momentum in the
#      Nifty 500". This belongs in the log's data_and_universe field.
#
#   4. n_valid_returns IS RETAINED so the 186 (80%) diagnostic is a FILTER ON A
#      COLUMN, not a second independent computation that could drift.
#
# NOTHING IS WRITTEN. Read-only.
# =============================================================================

LOOKBACK = 252   # trading rows back from rebalance -> window start
SKIP = 21        # trading rows skipped immediately before rebalance
WINDOW_LEN = LOOKBACK - SKIP   # 231 intervals... see note below
STRICT_MIN = 232               # required valid returns under your hard filter
RELAXED_MIN = 186              # 80% of 232 -- DIAGNOSTIC ONLY, not a result

print("--- Momentum signal specification ---")
print(f"  LOOKBACK      = {LOOKBACK} trading rows")
print(f"  SKIP          = {SKIP} trading rows")
print(f"  STRICT_MIN    = {STRICT_MIN} valid returns required (your decision)")
print(f"  RELAXED_MIN   = {RELAXED_MIN} (diagnostic comparison only)")

# -----------------------------------------------------------------------------
# WINDOW CONVENTION -- stated explicitly because off-by-one here is the single
# most likely silent error in this whole notebook.
#
# For a ticker whose rows are indexed 0..N-1 in date order, and a rebalance
# falling at positional index p:
#
#     window = rows [p - LOOKBACK, p - SKIP]   INCLUSIVE on both ends
#
# That is (LOOKBACK - SKIP + 1) = 232 rows.
# The most recent row in the window is p-21, i.e. 21 trading rows before the
# rebalance. Row p itself and rows p-1 .. p-20 are excluded (the skip window).
# -----------------------------------------------------------------------------
EXPECTED_WINDOW_ROWS = LOOKBACK - SKIP + 1
print(f"  Window spans rows [p-{LOOKBACK}, p-{SKIP}] inclusive "
      f"= {EXPECTED_WINDOW_ROWS} rows")
assert EXPECTED_WINDOW_ROWS == STRICT_MIN, (
    f"Window length {EXPECTED_WINDOW_ROWS} != STRICT_MIN {STRICT_MIN}. "
    "These must agree or the strict filter is not actually 'full window'."
)


# -----------------------------------------------------------------------------
# Per-ticker positional lookup.
# Build once: for each ticker, an array of its dates and its daily returns in
# date order, plus a map from date -> positional index.
# -----------------------------------------------------------------------------
print("\n--- Building per-ticker positional arrays ---")

ticker_data = {}
for tkr, grp in panel.groupby(TICKER_COL, sort=False):
    # grp is already date-sorted because panel was sorted canonically in A0.
    dates = grp[DATE_COL].to_numpy()
    rets = grp["daily_return"].to_numpy(dtype=float)
    # Normalise to a single EXPLICIT unit so every downstream comparison is
    # unit-consistent.
    #
    # The previous implementation kept a dict {date -> position}. That was a
    # 100%-failure bug: np.datetime64 hashes by UNIT as well as by value, so a
    # datetime64[ns] key and a datetime64[us] lookup silently miss each other
    # even when they denote the same instant. Every lookup returned None and
    # every signal came back NaN with reason 'no_row_on_rebalance_date'.
    #
    # searchsorted compares by VALUE, which removes the failure mode rather
    # than patching around it.
    # Boolean mask: True where this row falls on a VENDOR-FAILURE date. Used to
    # exclude those rows from the strict-coverage requirement (see the note in
    # compute_momentum_signal).
    is_vf = grp[DATE_COL].isin(VENDOR_FAILURE_DATES).to_numpy(dtype=bool)
    ticker_data[tkr] = {
        "dates": dates.astype("datetime64[ns]"),
        "returns": rets,
        "is_vendor_failure": is_vf,
    }

print(f"  Built arrays for {len(ticker_data):,} tickers")


def _position_of(date_array, as_of):
    """Positional index of `as_of` within a sorted datetime64[ns] array.

    Returns None if that exact date is not present. Uses searchsorted (value
    comparison) rather than a hash lookup, because np.datetime64 hashes by unit
    as well as value -- see the note in the ticker_data build above.
    """
    target = np.datetime64(pd.Timestamp(as_of), "ns")
    idx = int(np.searchsorted(date_array, target))
    if idx < len(date_array) and date_array[idx] == target:
        return idx
    return None


def compute_momentum_signal(ticker, as_of, lookback=LOOKBACK, skip=SKIP):
    """Momentum signal for one ticker as of one rebalance date.

    The `as_of` parameter is EXPLICIT and REQUIRED -- the point-in-time
    assumption is part of the function signature, not a trust-me comment.

    Returns a dict so the caller gets the diagnostics alongside the value:
        signal          compounded return over the window, or NaN
        n_valid         count of non-NaN returns actually in the window
        window_start    date of the first row in the window (or NaT)
        window_end      date of the last row in the window (or NaT)
        reason          why the signal is NaN, if it is
    """
    td = ticker_data.get(ticker)
    null_result = {"signal": np.nan, "n_valid": 0, "n_valid_counted": 0,
                   "n_required": EXPECTED_WINDOW_ROWS, "n_excluded_vendor": 0,
                   "window_start": pd.NaT, "window_end": pd.NaT,
                   "reason": "ticker_absent"}
    if td is None:
        return null_result

    # Value-based positional lookup. searchsorted returns the insertion point;
    # we then verify the date AT that position is exactly the rebalance date.
    # If it is not, the ticker genuinely did not trade that day (or the date
    # predates its listing) -- a real ineligibility, not a lookup failure.
    p = _position_of(td["dates"], as_of)
    if p is None:
        # Ticker did not trade on the rebalance date. It has no observable
        # price at the moment we would form the portfolio, so it is ineligible.
        return {**null_result, "reason": "no_row_on_rebalance_date"}

    start = p - lookback
    end = p - skip  # INCLUSIVE

    if start < 0:
        # Insufficient history -- expected for every ticker in the first ~year,
        # and for genuinely new listings thereafter.
        return {**null_result, "reason": "insufficient_history"}

    window_rets = td["returns"][start:end + 1]
    window_dates = td["dates"][start:end + 1]
    window_vf = td["is_vendor_failure"][start:end + 1]

    # Structural assert: the window must be exactly the expected length.
    # If slicing ever produces something else, we want a loud failure.
    if len(window_rets) != EXPECTED_WINDOW_ROWS:
        return {**null_result,
                "reason": f"window_len_{len(window_rets)}_expected_{EXPECTED_WINDOW_ROWS}"}

    # -------------------------------------------------------------------------
    # VENDOR-FAILURE ADJUSTMENT.
    #
    # 2025-03-18 was a LIVE NSE trading day on which yfinance failed to deliver
    # data for 896 of 915 tickers. It is retained in the panel (deleting a real
    # session would splice returns across a day that actually happened) and is
    # tagged `date_status == 'vendor_failure'`.
    #
    # A vendor outage is not a property of the STOCK. Requiring 232/232 valid
    # returns across a window spanning that date would reject nearly every name
    # for something the data vendor did -- the identical failure mode the six
    # fabricated NSE holidays caused, at smaller scale.
    #
    # So the rule becomes what it always meant: every return in the window must
    # be valid EXCEPT on vendor-failure dates, which drop out of the
    # denominator. `n_required` shrinks by the number of such rows in THIS
    # window, so a window that does not span one is unaffected.
    #
    # NOTE ON THE SUCCESSOR ROW: a vendor-failure date kills TWO returns -- its
    # own (no close) and the next session's (no prior close). Only the first is
    # tagged in the panel. The successor is derived structurally here rather
    # than hardcoded, so this stays correct if the tagging ever changes.
    # -------------------------------------------------------------------------
    excluded = window_vf.copy()
    if excluded.any():
        succ = np.zeros_like(window_vf)
        succ[1:] = window_vf[:-1]
        excluded = excluded | succ

    n_excluded = int(excluded.sum())
    n_required = EXPECTED_WINDOW_ROWS - n_excluded

    counted = window_rets[~excluded]
    n_valid_counted = int(np.sum(~np.isnan(counted)))
    n_valid_raw = int(np.sum(~np.isnan(window_rets)))

    if n_valid_counted < n_required:
        return {"signal": np.nan, "n_valid": n_valid_raw,
                "n_valid_counted": n_valid_counted, "n_required": n_required,
                "n_excluded_vendor": n_excluded,
                "window_start": pd.Timestamp(window_dates[0]),
                "window_end": pd.Timestamp(window_dates[-1]),
                "reason": "insufficient_valid_returns"}

    # Compound over the rows that ARE valid. Excluded rows are NaN by
    # construction, so they drop out of the product rather than contributing a
    # zero return -- a zero would be a FABRICATED value, and this project never
    # substitutes a plausible number for an unknown one.
    good = window_rets[~np.isnan(window_rets)]
    sig = float(np.prod(1.0 + good) - 1.0)
    return {"signal": sig, "n_valid": n_valid_raw,
            "n_valid_counted": n_valid_counted, "n_required": n_required,
            "n_excluded_vendor": n_excluded,
            "window_start": pd.Timestamp(window_dates[0]),
            "window_end": pd.Timestamp(window_dates[-1]),
            "reason": "ok"}


# -----------------------------------------------------------------------------
# Compute the signal for every (ticker, rebalance_date) pair.
# We compute for ALL tickers, not just flagged ones -- the PIT filter is applied
# in A4. Keeping them separate means we can see how much each filter removes.
# -----------------------------------------------------------------------------
print("\n--- Computing momentum signal across all (ticker, rebalance) pairs ---")
print(f"  {len(ticker_data):,} tickers x {N_REBALANCES} rebalances "
      f"= {len(ticker_data) * N_REBALANCES:,} pairs. This takes a few minutes.")

rows = []
for ri, rdate in enumerate(REBALANCE_DATES):
    if ri % 24 == 0:
        print(f"    ... rebalance {ri + 1}/{N_REBALANCES}  ({rdate.date()})")
    for tkr in ticker_data:
        res = compute_momentum_signal(tkr, rdate)
        rows.append({
            "rebalance_date": rdate,
            TICKER_COL: tkr,
            "momentum_signal_strict": res["signal"],
            "n_valid_returns": res["n_valid"],
            "n_valid_counted": res["n_valid_counted"],
            "n_required": res["n_required"],
            "n_excluded_vendor": res["n_excluded_vendor"],
            "window_start": res["window_start"],
            "window_end": res["window_end"],
            "signal_reason": res["reason"],
        })

signals = pd.DataFrame(rows)
print(f"\n  Signal frame built: {len(signals):,} rows")

# Attach the PIT flag and turnover as of the rebalance date.
# Merge on (ticker, date) -- an inner-key left join so a missing panel row
# surfaces as NaN rather than silently dropping the pair.
_panel_slim = panel[[TICKER_COL, DATE_COL, "in_nifty500_flag", "turnover",
                     "price_status"]].rename(columns={DATE_COL: "rebalance_date"})
signals = signals.merge(_panel_slim, on=[TICKER_COL, "rebalance_date"], how="left")
print(f"  After merging flag/turnover: {len(signals):,} rows "
      f"(must be unchanged -- merge must not fan out)")
check("flag/turnover merge did not change row count",
      len(signals) == len(rows),
      f"before={len(rows):,} after={len(signals):,}")


# -----------------------------------------------------------------------------
# COVERAGE DIAGNOSTIC
# -----------------------------------------------------------------------------
print("\n--- Coverage: strict (232) vs relaxed (186) ---")

signals["has_strict"] = signals["momentum_signal_strict"].notna()
# The relaxed count is a FILTER ON n_valid_returns, not a recomputation.
# Note: a pair with n_valid >= 186 but < 232 has NO strict signal value; we are
# only counting how many MORE names would qualify, not computing their signal.
# Relaxed comparison applies the SAME vendor-failure adjustment, scaled: 80% of
# whatever this window actually required. Comparing an adjusted strict count
# against an unadjusted relaxed one would overstate the gap.
signals["would_pass_relaxed"] = (
    signals["n_valid_counted"] >= (0.80 * signals["n_required"]).round())

signals["year"] = signals["rebalance_date"].dt.year

coverage_by_year = (
    signals.groupby("year")
           .agg(ticker_months=("has_strict", "size"),
                strict_valid=("has_strict", "sum"),
                relaxed_valid=("would_pass_relaxed", "sum"))
           .reset_index()
)
coverage_by_year["strict_pct"] = 100 * coverage_by_year["strict_valid"] / coverage_by_year["ticker_months"]
coverage_by_year["relaxed_pct"] = 100 * coverage_by_year["relaxed_valid"] / coverage_by_year["ticker_months"]
coverage_by_year["extra_from_relaxing"] = coverage_by_year["relaxed_valid"] - coverage_by_year["strict_valid"]

print("\n  Coverage by year (denominator = all tickers x rebalances that year):")
print(f"  {'year':<6} {'ticker_mo':>10} {'strict':>9} {'strict%':>9} "
      f"{'relaxed':>9} {'relax%':>9} {'extra':>8}")
for _, r in coverage_by_year.iterrows():
    print(f"  {int(r['year']):<6} {int(r['ticker_months']):>10,} "
          f"{int(r['strict_valid']):>9,} {r['strict_pct']:>8.1f}% "
          f"{int(r['relaxed_valid']):>9,} {r['relaxed_pct']:>8.1f}% "
          f"{int(r['extra_from_relaxing']):>8,}")

_overall_strict = 100 * signals["has_strict"].sum() / len(signals)
_overall_relaxed = 100 * signals["would_pass_relaxed"].sum() / len(signals)
_vf_aff = signals[signals["n_excluded_vendor"] > 0]
print(f"\n  Ticker-months whose window spans a vendor-failure date: "
      f"{len(_vf_aff):,} ({100*len(_vf_aff)/len(signals):.2f}%)")
if len(_vf_aff):
    print(f"    rows excluded from requirement: "
          f"{dict(_vf_aff['n_excluded_vendor'].value_counts().sort_index())}")
    _vf_ok = int((_vf_aff["signal_reason"] == "ok").sum())
    print(f"    of these, {_vf_ok:,} produced a valid signal "
          f"({100*_vf_ok/len(_vf_aff):.1f}%)")
    print("    WITHOUT this adjustment all of them would have failed 232/232.")

print(f"\n  OVERALL strict coverage:  {_overall_strict:.2f}%")
print(f"  OVERALL relaxed coverage: {_overall_relaxed:.2f}%")
print(f"  Relaxing 232 -> 186 would add "
      f"{int(signals['would_pass_relaxed'].sum() - signals['has_strict'].sum()):,} "
      f"ticker-months ({_overall_relaxed - _overall_strict:+.2f}pp)")

# -----------------------------------------------------------------------------
# Why are signals NaN? Attribution matters -- "insufficient_valid_returns" is a
# data-quality exclusion, "insufficient_history" is a structural warm-up.
# -----------------------------------------------------------------------------
print("\n--- NaN attribution (why a signal is missing) ---")
reason_counts = signals["signal_reason"].value_counts()
for reason, cnt in reason_counts.items():
    print(f"  {reason:<36} {cnt:>10,}  ({100*cnt/len(signals):>5.2f}%)")

# -----------------------------------------------------------------------------
# Names dropped per month by the strict filter, among names that HAD a full
# 232-row window available (i.e. excluding structural warm-up).
# This isolates the data-quality cost of the strict filter from the warm-up cost.
# -----------------------------------------------------------------------------
print("\n--- Strict-filter drop rate among names WITH a full window available ---")
_had_window = signals[signals["signal_reason"].isin(["ok", "insufficient_valid_returns"])].copy()
drop_by_month = (
    _had_window.groupby("rebalance_date")
               .agg(had_window=("has_strict", "size"),
                    passed_strict=("has_strict", "sum"))
               .reset_index()
)
drop_by_month["dropped"] = drop_by_month["had_window"] - drop_by_month["passed_strict"]
drop_by_month["drop_pct"] = 100 * drop_by_month["dropped"] / drop_by_month["had_window"].replace(0, np.nan)
drop_by_month["year"] = drop_by_month["rebalance_date"].dt.year

drop_by_year = drop_by_month.groupby("year").agg(
    mean_had_window=("had_window", "mean"),
    mean_passed=("passed_strict", "mean"),
    mean_dropped=("dropped", "mean"),
    mean_drop_pct=("drop_pct", "mean"),
).reset_index()

print(f"  {'year':<6} {'avg_window_avail':>18} {'avg_passed':>12} "
      f"{'avg_dropped':>13} {'avg_drop%':>11}")
for _, r in drop_by_year.iterrows():
    print(f"  {int(r['year']):<6} {r['mean_had_window']:>18.1f} "
          f"{r['mean_passed']:>12.1f} {r['mean_dropped']:>13.1f} "
          f"{r['mean_drop_pct']:>10.1f}%")

# -----------------------------------------------------------------------------
# Fact #22 predicted thin coverage in 2014 and 2025 for realized vol. Momentum
# needs 232 continuous returns, so the same churn should show up here.
# We MEASURE rather than assume.
# -----------------------------------------------------------------------------
print("\n--- Fact #22 cross-check: is 2014 / 2025 coverage anomalously thin? ---")
_cb = coverage_by_year.set_index("year")
_median_strict_pct = coverage_by_year.loc[
    coverage_by_year["year"].between(2012, 2024), "strict_pct"].median()
print(f"  Median strict coverage 2012-2024: {_median_strict_pct:.1f}%")
for yr in (2014, 2025):
    if yr in _cb.index:
        v = _cb.loc[yr, "strict_pct"]
        flagmark = "  <-- THIN" if v < 0.75 * _median_strict_pct else ""
        print(f"  {yr}: {v:.1f}%{flagmark}")
        RUN_LOG.append((f"strict coverage {yr}", "INFO", f"{v:.1f}%"))

print("\n" + "=" * 78)
print("CELL A2 COMPLETE: signal frame built, coverage measured")
print("=" * 78)

--- Momentum signal specification ---
  LOOKBACK      = 252 trading rows
  SKIP          = 21 trading rows
  STRICT_MIN    = 232 valid returns required (your decision)
  RELAXED_MIN   = 186 (diagnostic comparison only)
  Window spans rows [p-252, p-21] inclusive = 232 rows

--- Building per-ticker positional arrays ---
  Built arrays for 1,069 tickers

--- Computing momentum signal across all (ticker, rebalance) pairs ---
  1,069 tickers x 199 rebalances = 212,731 pairs. This takes a few minutes.
    ... rebalance 1/199  (2010-01-29)
    ... rebalance 25/199  (2012-01-31)
    ... rebalance 49/199  (2014-01-31)
    ... rebalance 73/199  (2016-01-29)
    ... rebalance 97/199  (2018-01-31)
    ... rebalance 121/199  (2020-01-31)
    ... rebalance 145/199  (2022-01-31)
    ... rebalance 169/199  (2024-01-31)
    ... rebalance 193/199  (2026-01-30)

  Signal frame built: 212,731 rows
  After merging flag/turnover: 212,731 rows (must be unchanged -- merge must not fan out)
  [PASS] flag/turn

## Cell A3 — Leakage proof + manual spot-check

Discharges validation requirements **#2** and **#3** together.

For three pairs chosen by *position* in the sample (early/mid/late) rather than
by signal value — picking by value would bias toward well-behaved names — this
prints the exact window date range, the skip window, and a compounding trace of
the first and last five returns so the arithmetic can be **read rather than
trusted**. The manual recomputation goes back to `panel` rather than reusing
`ticker_data`, so it is a genuine second path through the data.

The assertion that matters: `max(window_dates) < rebalance_date` with exactly 21
trading rows between them. This is what catches an off-by-one where the window
ends at `t-20` or `t-22` — a bug that compiles fine and is nearly invisible in
aggregate results.

A **global** version of the same assertion then runs across every computed
signal, which is what would catch a bug affecting only tickers with gaps.

In [4]:
# =============================================================================
# CELL A3 -- Leakage proof + manual spot-check
# =============================================================================
# This cell discharges TWO mandatory validation requirements at once:
#
#   (#3) LEAKAGE CHECK, EXPLICIT -- we print the exact date range entering the
#        window for sample pairs and assert max(window) < rebalance_date with a
#        gap of exactly SKIP trading rows. This is the check that catches an
#        off-by-one where the window ends at t-20 or t-22 -- a bug that would
#        compile fine and be nearly invisible in aggregate results.
#
#   (#2) MANUAL SPOT-CHECK -- we independently recompute the signal for the
#        same pairs by pulling raw returns and compounding them in a visible
#        printed table, so the arithmetic can be READ rather than trusted.
#
# NOTHING IS WRITTEN. Read-only.
# =============================================================================

print("--- Selecting spot-check pairs ---")

# Pick 3 pairs with a valid strict signal, spread across the sample period.
# Deliberately chosen by position (early / middle / late) rather than by signal
# value -- picking by value would bias us toward well-behaved names.
_valid = signals[signals["signal_reason"] == "ok"].copy()
_valid = _valid.sort_values(["rebalance_date", TICKER_COL]).reset_index(drop=True)

if len(_valid) < 3:
    raise RuntimeError(
        f"Only {len(_valid)} valid signals exist -- cannot spot-check. "
        "Investigate A2 before proceeding."
    )

_pick_positions = [
    int(len(_valid) * 0.05),   # early sample
    int(len(_valid) * 0.50),   # mid sample
    int(len(_valid) * 0.92),   # late sample
]
SPOT_CHECKS = []
for pos in _pick_positions:
    r = _valid.iloc[pos]
    SPOT_CHECKS.append((r[TICKER_COL], r["rebalance_date"]))

print(f"  Selected {len(SPOT_CHECKS)} pairs:")
for t, d in SPOT_CHECKS:
    print(f"    {t}  @  {d.date()}")


SPOT_CHECK_RESULTS = []

for tkr, rdate in SPOT_CHECKS:
    print("\n" + "-" * 78)
    print(f"SPOT-CHECK: {tkr}  as of rebalance {rdate.date()}")
    print("-" * 78)

    # -------------------------------------------------------------------------
    # INDEPENDENT RECOMPUTATION.
    # We deliberately go back to `panel` here rather than reusing ticker_data,
    # so this is a genuine second path through the data, not a re-read of the
    # same arrays the vectorised version used.
    # -------------------------------------------------------------------------
    sub = panel[panel[TICKER_COL] == tkr].sort_values(DATE_COL).reset_index(drop=True)
    p = sub.index[sub[DATE_COL] == rdate]
    assert len(p) == 1, f"Expected exactly 1 row for {tkr} on {rdate.date()}, got {len(p)}"
    p = int(p[0])

    start = p - LOOKBACK
    end = p - SKIP
    win = sub.iloc[start:end + 1]

    manual_rets = win["daily_return"].to_numpy(dtype=float)

    # The manual recomputation must apply the SAME vendor-failure treatment as
    # the vectorised version, or the two disagree by construction on any window
    # spanning 2025-03-18. Excluded rows are NaN, and NaN rows are DROPPED from
    # the product (never treated as a zero return -- a zero would be fabricated).
    manual_good = manual_rets[~np.isnan(manual_rets)]
    manual_signal = float(np.prod(1.0 + manual_good) - 1.0)
    n_manual_nan = int(np.isnan(manual_rets).sum())
    if n_manual_nan:
        print(f"  NOTE: {n_manual_nan} NaN return(s) in this window "
              f"(vendor-failure exclusion) — dropped from the product, "
              f"so {len(manual_good)} terms compound, not {len(manual_rets)}.")

    # The vectorised value from A2:
    vec_row = signals[(signals[TICKER_COL] == tkr) &
                      (signals["rebalance_date"] == rdate)].iloc[0]
    vec_signal = float(vec_row["momentum_signal_strict"])

    # -------------------------------------------------------------------------
    # LEAKAGE ASSERTIONS -- printed, not just asserted, so they can be read.
    # -------------------------------------------------------------------------
    w_first = win[DATE_COL].iloc[0]
    w_last = win[DATE_COL].iloc[-1]
    n_rows_in_window = len(win)

    # How many trading rows separate the last window row from the rebalance?
    rows_between = p - end   # should equal SKIP exactly

    # Which rows are excluded by the skip window?
    skipped = sub.iloc[end + 1:p + 1]

    print(f"  Rebalance date            : {rdate.date()}")
    print(f"  Window FIRST date         : {w_first.date()}")
    print(f"  Window LAST date          : {w_last.date()}")
    print(f"  Rows in window            : {n_rows_in_window}  (expected {EXPECTED_WINDOW_ROWS})")
    print(f"  Trading rows from window-end to rebalance : {rows_between}  (expected {SKIP})")
    print(f"  Calendar days window-end -> rebalance     : {(rdate - w_last).days}")
    print(f"  Skipped rows (window_end+1 .. rebalance)  : {len(skipped)}")
    print(f"    skip window spans {skipped[DATE_COL].iloc[0].date()} "
          f"-> {skipped[DATE_COL].iloc[-1].date()}")

    a1 = w_last < rdate
    a2 = rows_between == SKIP
    a3 = n_rows_in_window == EXPECTED_WINDOW_ROWS
    a4 = win[DATE_COL].max() < rdate
    a5 = len(skipped) == SKIP

    print(f"\n  LEAKAGE ASSERTIONS:")
    print(f"    [{'PASS' if a1 else 'FAIL'}] window_last < rebalance_date")
    print(f"    [{'PASS' if a2 else 'FAIL'}] exactly {SKIP} trading rows skipped")
    print(f"    [{'PASS' if a3 else 'FAIL'}] window contains exactly {EXPECTED_WINDOW_ROWS} rows")
    print(f"    [{'PASS' if a4 else 'FAIL'}] max(window dates) < rebalance_date")
    print(f"    [{'PASS' if a5 else 'FAIL'}] skip window is exactly {SKIP} rows")

    check(f"leakage/{tkr}@{rdate.date()}: window strictly before rebalance", a1 and a4)
    check(f"leakage/{tkr}@{rdate.date()}: skip is exactly {SKIP} rows", a2 and a5)
    check(f"leakage/{tkr}@{rdate.date()}: window length {EXPECTED_WINDOW_ROWS}", a3)

    # -------------------------------------------------------------------------
    # VISIBLE ARITHMETIC -- first 5 and last 5 returns, with running compound,
    # so the compounding can be followed by hand.
    # -------------------------------------------------------------------------
    print(f"\n  Compounding trace (first 5 and last 5 of {n_rows_in_window} rows):")
    print(f"    {'date':<12} {'daily_return':>14} {'running (1+r) prod':>22}")
    running = 1.0
    for i in range(n_rows_in_window):
        if not np.isnan(manual_rets[i]):
            running *= (1.0 + manual_rets[i])
        if i < 5 or i >= n_rows_in_window - 5:
            print(f"    {str(win[DATE_COL].iloc[i].date()):<12} "
                  f"{manual_rets[i]:>14.8f} {running:>22.10f}")
        elif i == 5:
            print(f"    {'...':<12} {'...':>14} {'...':>22}")

    print(f"\n  MANUAL     signal = {manual_signal:.10f}")
    print(f"  VECTORISED signal = {vec_signal:.10f}")
    print(f"  Absolute difference = {abs(manual_signal - vec_signal):.3e}")

    agree = np.isclose(manual_signal, vec_signal, rtol=1e-12, atol=1e-12)
    check(f"spotcheck/{tkr}@{rdate.date()}: manual == vectorised",
          agree, f"diff={abs(manual_signal - vec_signal):.3e}")

    SPOT_CHECK_RESULTS.append({
        "ticker": tkr,
        "rebalance_date": rdate,
        "window_start": w_first,
        "window_end": w_last,
        "n_rows": n_rows_in_window,
        "rows_skipped": len(skipped),
        "manual_signal": manual_signal,
        "vectorised_signal": vec_signal,
        "abs_diff": abs(manual_signal - vec_signal),
        "agree": bool(agree),
    })

spot_check_df = pd.DataFrame(SPOT_CHECK_RESULTS)

# -----------------------------------------------------------------------------
# GLOBAL leakage assertion across EVERY computed signal, not just the 3 samples.
# The spot-checks prove the arithmetic; this proves the property holds
# universally. Cheap, and it is the check that would catch a bug affecting only
# some tickers (e.g. those with gaps).
# -----------------------------------------------------------------------------
print("\n" + "-" * 78)
print("GLOBAL leakage assertion across all computed signals")
print("-" * 78)

_ok = signals[signals["signal_reason"] == "ok"]
_violations = _ok[_ok["window_end"] >= _ok["rebalance_date"]]
check("GLOBAL: no signal window ends on/after its rebalance date",
      len(_violations) == 0,
      f"violations={len(_violations):,} of {len(_ok):,}")

if len(_violations) > 0:
    print("\n  FIRST 10 VIOLATIONS -- this is a leakage bug, stop and fix:")
    print(_violations[[TICKER_COL, "rebalance_date", "window_start",
                       "window_end"]].head(10).to_string(index=False))

# Distribution of the calendar gap between window end and rebalance.
# Should cluster around ~30 calendar days (21 trading days), widening around
# holiday clusters. A gap near zero would indicate the skip is not applied.
_gaps = (_ok["rebalance_date"] - _ok["window_end"]).dt.days
print(f"\n  Calendar-day gap (window_end -> rebalance) across {len(_ok):,} signals:")
print(f"    min={_gaps.min()}  p01={_gaps.quantile(0.01):.0f}  "
      f"median={_gaps.median():.0f}  p99={_gaps.quantile(0.99):.0f}  max={_gaps.max()}")
check("GLOBAL: minimum calendar gap >= 21 days (skip window is real)",
      _gaps.min() >= 21, f"min_gap={_gaps.min()} days")

print("\n" + "=" * 78)
print("CELL A3 COMPLETE: leakage proof + spot-checks done")
print("=" * 78)

--- Selecting spot-check pairs ---
  Selected 3 pairs:
    GIPCL.NS  @  2011-12-30
    MOTHERSON.NS  @  2019-06-28
    JISLJALEQS.NS  @  2025-07-31

------------------------------------------------------------------------------
SPOT-CHECK: GIPCL.NS  as of rebalance 2011-12-30
------------------------------------------------------------------------------
  Rebalance date            : 2011-12-30
  Window FIRST date         : 2010-12-23
  Window LAST date          : 2011-11-30
  Rows in window            : 232  (expected 232)
  Trading rows from window-end to rebalance : 21  (expected 21)
  Calendar days window-end -> rebalance     : 30
  Skipped rows (window_end+1 .. rebalance)  : 21
    skip window spans 2011-12-01 -> 2011-12-30

  LEAKAGE ASSERTIONS:
    [PASS] window_last < rebalance_date
    [PASS] exactly 21 trading rows skipped
    [PASS] window contains exactly 232 rows
    [PASS] max(window dates) < rebalance_date
    [PASS] skip window is exactly 21 rows
  [PASS] leakage/GIPCL.N

## Cell A4 — PIT universe check

The filter is `in_nifty500_flag == 1`. **Not** `!= 0`, **not** `.astype(bool)`,
**not** `.fillna(False)`.

Fact #17: the flag is tri-state. `NaN` means *unknown* (pre-2010-06-30), not
"not a member". `!= 1` silently bundles the ~85,464 unknowns with the known-outs;
`!= 0` bundles them with the known-ins. Both are wrong. The three states are
counted separately so the unknowns stay visible.

**Trip-wire:** if the eligible universe is nearly constant near 500 across all
years, the filter probably is not being applied — real PIT membership plus a
strict data filter must show visible variation. The cell prints the standard
deviation for exactly this reason.

Generic statistics cannot prove point-in-time correctness. The named RANBAXY and
HDFC checks can. These **print rather than hard-assert**, because fact #18 means
membership resolves only to the semi-annual snapshot grid — up to ~6 months of
slack is expected, and asserting an exact date would produce a false failure.

In [5]:
# =============================================================================
# CELL A4 -- PIT universe filter check
# =============================================================================
# Discharges mandatory validation requirement (#4): UNIVERSE CHECK, EXPLICIT.
#
# THE FILTER: in_nifty500_flag == 1
#
#   NOT `!= 0`. NOT `astype(bool)`. NOT `.fillna(False)`.
#   Fact #17: the flag is TRI-STATE (1 / 0 / NaN). NaN means UNKNOWN
#   (pre-2010-06-30), not "not a member". Using `!= 1` would silently bundle
#   the ~85,464 unknowns with the known-outs; using `!= 0` would bundle them
#   with the known-ins. Both are wrong. We use `== 1` and treat NaN as
#   ineligible-but-distinct, and we COUNT the NaN cases so they are visible.
#
# NOTHING IS WRITTEN. Read-only.
# =============================================================================

print("--- Applying PIT universe filter: in_nifty500_flag == 1 ---")

# Three mutually exclusive states, counted explicitly.
signals["flag_state"] = np.select(
    [signals["in_nifty500_flag"] == 1,
     signals["in_nifty500_flag"] == 0,
     signals["in_nifty500_flag"].isna()],
    ["in", "out", "unknown"],
    default="UNEXPECTED",
)

_unexpected = int((signals["flag_state"] == "UNEXPECTED").sum())
check("no unexpected flag values in signal frame", _unexpected == 0,
      f"count={_unexpected:,}")

print("\n  Flag state distribution across all (ticker, rebalance) pairs:")
for st, cnt in signals["flag_state"].value_counts().items():
    print(f"    {st:<12} {cnt:>10,}  ({100*cnt/len(signals):>5.2f}%)")

# THE ELIGIBLE SET: flagged in AND has a valid strict signal.
signals["is_eligible"] = (signals["flag_state"] == "in") & signals["has_strict"]

print(f"\n  Eligible (flag==1 AND strict signal valid): "
      f"{int(signals['is_eligible'].sum()):,} ticker-months")

# -----------------------------------------------------------------------------
# Universe size per rebalance -- flagged vs eligible.
# The gap between them is the cost of the 232 filter.
# -----------------------------------------------------------------------------
print("\n--- Universe size by year: flagged vs eligible ---")

univ_by_reb = (
    signals.groupby("rebalance_date")
           .agg(n_flagged_in=("flag_state", lambda s: int((s == "in").sum())),
                n_eligible=("is_eligible", "sum"))
           .reset_index()
)
univ_by_reb["n_lost_to_signal_filter"] = (
    univ_by_reb["n_flagged_in"] - univ_by_reb["n_eligible"]
)
univ_by_reb["year"] = univ_by_reb["rebalance_date"].dt.year

univ_by_year = univ_by_reb.groupby("year").agg(
    mean_flagged=("n_flagged_in", "mean"),
    min_flagged=("n_flagged_in", "min"),
    max_flagged=("n_flagged_in", "max"),
    mean_eligible=("n_eligible", "mean"),
    min_eligible=("n_eligible", "min"),
    max_eligible=("n_eligible", "max"),
).reset_index()
univ_by_year["mean_lost"] = univ_by_year["mean_flagged"] - univ_by_year["mean_eligible"]
univ_by_year["pct_retained"] = 100 * univ_by_year["mean_eligible"] / univ_by_year["mean_flagged"]

print(f"  {'year':<6} {'flagged(avg)':>13} {'elig(avg)':>11} {'elig(min)':>11} "
      f"{'elig(max)':>11} {'lost':>8} {'retained%':>11}")
for _, r in univ_by_year.iterrows():
    print(f"  {int(r['year']):<6} {r['mean_flagged']:>13.1f} {r['mean_eligible']:>11.1f} "
          f"{int(r['min_eligible']):>11,} {int(r['max_eligible']):>11,} "
          f"{r['mean_lost']:>8.1f} {r['pct_retained']:>10.1f}%")

# -----------------------------------------------------------------------------
# SANITY TRIP-WIRE.
# If the eligible universe is suspiciously stable near 500 across ALL years,
# the filter probably is not being applied at all. Real PIT membership plus a
# strict data filter should show visible variation.
# -----------------------------------------------------------------------------
_elig_std = univ_by_reb["n_eligible"].std()
_elig_mean = univ_by_reb["n_eligible"].mean()
print(f"\n  Eligible universe: mean={_elig_mean:.1f}  std={_elig_std:.1f}  "
      f"CV={_elig_std/_elig_mean if _elig_mean else float('nan'):.3f}")
if _elig_std < 5:
    print("  WARNING: eligible universe is nearly CONSTANT across the whole "
          "sample. Verify the PIT filter is actually varying -- this is what a "
          "silently-dropped universe filter looks like.")
RUN_LOG.append(("eligible universe std", "INFO", f"{_elig_std:.1f}"))

# Recent-window expectation: memory doc says ~495 flagged vs ~500 true
# (5 no-price constituents: Vedanta demerger x3, Malco Energy, Kwality Wall's).
_recent = univ_by_reb[univ_by_reb["rebalance_date"] >= "2025-01-01"]
if len(_recent):
    print(f"\n  Recent window (2025+) mean flagged: {_recent['n_flagged_in'].mean():.1f}")
    print(f"    Memory doc expects ~495 (5 no-price constituents logged).")
    check("recent flagged universe in 480-505 range",
          480 <= _recent["n_flagged_in"].mean() <= 505,
          f"mean={_recent['n_flagged_in'].mean():.1f}")


# -----------------------------------------------------------------------------
# NAMED GROUND-TRUTH CHECKS.
# Generic statistics cannot prove the filter is point-in-time. These can.
# -----------------------------------------------------------------------------
def named_membership_check(ticker_fragment, label, expect_in_around, expect_out_around):
    """Print a ticker's flag state around two dates where we know the truth.

    expect_in_around / expect_out_around are (start, end) date strings.
    We do NOT assert hard here for HDFC-type cases where the exact convention
    may differ -- we PRINT and let the human adjudicate, because fact #18 says
    the flag resolves only to the semi-annual snapshot grid (up to ~6 months of
    slack). Asserting a hard date would produce a false failure.
    """
    print("\n" + "-" * 78)
    print(f"NAMED CHECK: {label}")
    print("-" * 78)

    matches = [t for t in signals[TICKER_COL].unique()
               if ticker_fragment.upper() in str(t).upper()]
    if not matches:
        print(f"  Ticker matching '{ticker_fragment}' NOT FOUND in panel.")
        print(f"  Cannot perform this ground-truth check. Recording as SKIPPED.")
        RUN_LOG.append((f"named check {label}", "SKIP", "ticker not found"))
        return None

    print(f"  Matched ticker(s): {matches}")
    tkr = matches[0]
    sub = signals[signals[TICKER_COL] == tkr].sort_values("rebalance_date")

    print(f"\n  Flag state at rebalances in the EXPECT-IN window "
          f"({expect_in_around[0]} to {expect_in_around[1]}):")
    win_in = sub[(sub["rebalance_date"] >= expect_in_around[0]) &
                 (sub["rebalance_date"] <= expect_in_around[1])]
    for _, r in win_in.iterrows():
        print(f"    {r['rebalance_date'].date()}  flag={r['flag_state']:<8} "
              f"eligible={bool(r['is_eligible'])}  signal_reason={r['signal_reason']}")

    print(f"\n  Flag state at rebalances in the EXPECT-OUT window "
          f"({expect_out_around[0]} to {expect_out_around[1]}):")
    win_out = sub[(sub["rebalance_date"] >= expect_out_around[0]) &
                  (sub["rebalance_date"] <= expect_out_around[1])]
    for _, r in win_out.iterrows():
        print(f"    {r['rebalance_date'].date()}  flag={r['flag_state']:<8} "
              f"eligible={bool(r['is_eligible'])}  signal_reason={r['signal_reason']}")

    n_in = int((win_in["flag_state"] == "in").sum())
    n_out_still_in = int((win_out["flag_state"] == "in").sum())
    print(f"\n  Summary: {n_in}/{len(win_in)} rebalances flagged IN during "
          f"expect-in window; {n_out_still_in}/{len(win_out)} still flagged IN "
          f"during expect-out window.")
    print(f"  NOTE (fact #18): membership resolves only to the semi-annual "
          f"snapshot grid, so up to ~6 months of slack is EXPECTED, not a bug.")

    RUN_LOG.append((f"named check {label}", "INFO",
                    f"in-window {n_in}/{len(win_in)} IN; "
                    f"out-window {n_out_still_in}/{len(win_out)} still IN"))
    return {"ticker": tkr, "n_in": n_in, "len_in": len(win_in),
            "n_out_still_in": n_out_still_in, "len_out": len(win_out)}


# RANBAXY: flagged out at 2014-12-31 per fact #18, traded until 2015-04-01.
ranbaxy_result = named_membership_check(
    "RANBAXY", "RANBAXY (expect IN during 2014, OUT by mid-2015)",
    expect_in_around=("2014-01-01", "2014-12-31"),
    expect_out_around=("2015-05-01", "2015-12-31"),
)

# HDFC: merged into HDFC Bank effective 2023-07-13.
hdfc_result = named_membership_check(
    "HDFC.NS", "HDFC (expect IN before 2023-07-12, OUT after merger)",
    expect_in_around=("2022-07-01", "2023-06-30"),
    expect_out_around=("2023-10-01", "2024-06-30"),
)

print("\n" + "=" * 78)
print("CELL A4 COMPLETE: PIT filter applied and ground-truth-checked")
print("=" * 78)

--- Applying PIT universe filter: in_nifty500_flag == 1 ---
  [PASS] no unexpected flag values in signal frame  --  count=0

  Flag state distribution across all (ticker, rebalance) pairs:
    in               93,394  (43.90%)
    out              60,375  (28.38%)
    unknown          58,962  (27.72%)

  Eligible (flag==1 AND strict signal valid): 85,184 ticker-months

--- Universe size by year: flagged vs eligible ---
  year    flagged(avg)   elig(avg)   elig(min)   elig(max)     lost   retained%
  2010           290.5         0.0           0           0    290.5        0.0%
  2011           455.9       428.3           0         483     27.6       93.9%
  2012           494.3       452.5         440         466     41.8       91.5%
  2013           493.5       470.4         464         480     23.1       95.3%
  2014           458.3       432.8           0         482     25.5       94.4%
  2015           461.8       438.6          56         486     23.2       95.0%
  2016           

## Cell A5 — Termination diagnostic  ⟵ **the decisive cell**

Measures first, then you decide — the agreed sequencing.

**Count alone is not enough; concentration is what matters.** Fifty terminations
spread evenly over sixteen years is noise. Fifty clustered in 2018 H2, or
clustered in the top momentum decile, is not — the latter especially, since a
stock that ran up hard and then stopped trading is exactly the case where the
exit assumption drives the spread.

Three-way classification, because conflating these would corrupt the count:

- `panel_edge` — series ends at the frozen 2026-07-10 boundary. **Not** a
  termination; counted separately or it inflates the number.
- `true_termination` — series ends with substantial panel history remaining.
- `masked_gap_resumes` — a NaN run that later resumes. Not an exit.

**Provisional deciles:** computed here purely to bucket terminations. No returns,
no portfolios, no spread. Phase B owns real decile construction — flagged so
decile logic appearing in Phase A is not read as scope creep.

Full detail goes to CSV; A7 prints the path.

In [6]:
# =============================================================================
# CELL A5 -- Mid-month termination diagnostic  [THE DECISIVE CELL]
# =============================================================================
# PURPOSE: count and characterise mid-month terminations BEFORE designing how
# to handle them. Per the agreed sequencing: measure first, then decide.
#
# WHAT MATTERS IS COUNT *AND* CONCENTRATION:
#   - N terminations spread evenly over 16 years is noise.
#   - N clustered in 2018 H2, or clustered in the TOP momentum decile, is not.
#     The latter especially: a stock that ran up hard and then stopped trading
#     is exactly the case where the exit assumption drives the spread.
#
# THREE-WAY CLASSIFICATION (conflating these would corrupt the count):
#   panel_edge     -- series ends at/near 2026-07-10. NOT a termination; the
#                     panel is frozen there. Counted separately.
#   true_termination -- series ends with substantial panel history remaining.
#   masked_gap_resumes -- a NaN run that LATER resumes. Not a termination.
#
# PROVISIONAL DECILES: this cell computes a decile rank purely to BUCKET
# terminations. No returns, no portfolios, no spread. Phase B owns real decile
# construction. Flagged so decile logic appearing here is not read as scope creep.
#
# WRITES ONE CSV (per-termination detail). No panel mutation.
# =============================================================================

print("--- Provisional decile ranks (for BUCKETING terminations only) ---")

elig = signals[signals["is_eligible"]].copy()
print(f"  Eligible ticker-months: {len(elig):,}")

# Cross-sectional decile within each rebalance date.
# method='first' breaks ties deterministically; Phase B will decide the real
# tie convention. qcut with duplicates='drop' would silently produce fewer than
# 10 buckets, so we rank-then-cut explicitly instead.
# NOTE: groupby(...).apply() can consume the grouping column. We assign into a
# pre-created column via transform-style indexing instead, which leaves `elig`
# structurally untouched. (This bug was caught in integration testing: the
# naive .apply() dropped `rebalance_date` from the columns and every downstream
# groupby on it raised KeyError.)
elig["prov_decile"] = np.nan
for _rd, _idx in elig.groupby("rebalance_date").groups.items():
    _grp = elig.loc[_idx]
    if len(_grp) < 10:
        continue
    _r = _grp["momentum_signal_strict"].rank(method="first", ascending=True)
    elig.loc[_idx, "prov_decile"] = np.ceil(10 * _r / len(_grp)).clip(1, 10)

_months_too_small = int(elig.groupby("rebalance_date")["prov_decile"]
                        .apply(lambda s: s.isna().all()).sum())
print(f"  Rebalance months with <10 eligible names (no deciles): {_months_too_small}")
print(f"  Decile distribution (should be near-uniform):")
_dd = elig["prov_decile"].value_counts().sort_index()
for d, c in _dd.items():
    print(f"    decile {int(d):>2}: {c:>8,}")

# -----------------------------------------------------------------------------
# FORWARD-MONTH TERMINATION DETECTION
#
# For each eligible (ticker, rebalance) pair, look at the forward holding month:
# rebalance_date (exclusive) -> next rebalance_date (inclusive).
# Ask: does the ticker have observable, valid returns through the whole month?
# -----------------------------------------------------------------------------
print("\n--- Detecting mid-month terminations ---")

reb_list = REBALANCE_DATES
next_reb = {reb_list[i]: reb_list[i + 1] for i in range(len(reb_list) - 1)}

# Panel edge tolerance: a series whose last observation is within this many
# trading days of the panel end is the frozen boundary, not a termination.
PANEL_EDGE_TOLERANCE_DAYS = 10
panel_edge_cutoff = PANEL_END - pd.Timedelta(days=PANEL_EDGE_TOLERANCE_DAYS)
print(f"  Panel end: {PANEL_END.date()}")
print(f"  Panel-edge cutoff: last obs after {panel_edge_cutoff.date()} "
      f"=> classified 'panel_edge', not a termination")

# Precompute each ticker's LAST date with a valid (non-NaN) daily_return.
_valid_rows = panel[panel["daily_return"].notna()]
last_valid_date = _valid_rows.groupby(TICKER_COL)[DATE_COL].max().to_dict()

termination_rows = []

for i, rdate in enumerate(reb_list):
    if rdate not in next_reb:
        continue  # final rebalance -- no forward month exists
    fwd_end = next_reb[rdate]

    month_elig = elig[elig["rebalance_date"] == rdate]
    if len(month_elig) == 0:
        continue

    for _, r in month_elig.iterrows():
        tkr = r[TICKER_COL]
        td = ticker_data[tkr]
        dates = td["dates"]
        rets = td["returns"]

        # Forward window: strictly after rebalance, up to and including fwd_end.
        mask = (dates > np.datetime64(rdate)) & (dates <= np.datetime64(fwd_end))
        fwd_dates = dates[mask]
        fwd_rets = rets[mask]

        n_fwd_rows = len(fwd_dates)
        n_fwd_valid = int(np.sum(~np.isnan(fwd_rets))) if n_fwd_rows else 0

        # How many trading days SHOULD there be? Use the market-wide count.
        market_mask = (TRADING_DAY_INDEX > rdate) & (TRADING_DAY_INDEX <= fwd_end)
        n_market_days = int(market_mask.sum())

        # Complete if the ticker has a valid return on essentially every
        # market day in the forward window.
        is_complete = (n_fwd_valid >= n_market_days)

        if is_complete:
            continue  # nothing to report

        # -- This pair did NOT have a complete forward month. Classify it. --
        lvd = last_valid_date.get(tkr, pd.NaT)
        lvd = pd.Timestamp(lvd) if pd.notna(lvd) else pd.NaT

        if pd.isna(lvd):
            klass = "no_valid_data_anywhere"
        elif lvd >= panel_edge_cutoff:
            # Series runs to the panel edge -- so any gap inside this forward
            # month is a masked gap, not a termination.
            klass = "masked_gap_resumes" if lvd > fwd_end else "panel_edge"
        elif lvd > fwd_end:
            # Ticker resumes trading AFTER this forward month -> gap, not exit.
            klass = "masked_gap_resumes"
        else:
            # Series genuinely ends on/before the end of this forward month,
            # with substantial panel history remaining afterwards.
            klass = "true_termination"

        termination_rows.append({
            "rebalance_date": rdate,
            TICKER_COL: tkr,
            "prov_decile": r["prov_decile"],
            "momentum_signal_strict": r["momentum_signal_strict"],
            "flag_state_at_rebalance": r["flag_state"],
            "forward_month_end": fwd_end,
            "n_market_days_in_month": n_market_days,
            "n_valid_returns_observed": n_fwd_valid,
            "days_held_observed": n_fwd_valid,
            "last_valid_date_anywhere": lvd,
            "classification": klass,
            "year": rdate.year,
        })

terminations = pd.DataFrame(termination_rows)
print(f"\n  Incomplete forward months detected: {len(terminations):,}")

if len(terminations) == 0:
    print("  NO incomplete forward months at all. This is surprising -- "
          "verify the detection logic before accepting it.")
    terminations = pd.DataFrame(columns=[
        "rebalance_date", TICKER_COL, "prov_decile", "momentum_signal_strict",
        "flag_state_at_rebalance", "forward_month_end", "n_market_days_in_month",
        "n_valid_returns_observed", "days_held_observed",
        "last_valid_date_anywhere", "classification", "year"])
    # NOTE: explicit columns=[...] -- fact #12. pd.DataFrame([]) on an empty
    # list yields ZERO columns and writes a 0-byte CSV that later crashes read.

# -----------------------------------------------------------------------------
# REPORTING
# -----------------------------------------------------------------------------
print("\n--- Terminations by classification ---")
if len(terminations):
    for k, c in terminations["classification"].value_counts().items():
        print(f"    {k:<26} {c:>8,}  ({100*c/len(terminations):>5.1f}%)")

TRUE_TERMS = terminations[terminations["classification"] == "true_termination"].copy() \
    if len(terminations) else terminations.copy()
N_TRUE_TERMS = len(TRUE_TERMS)

print(f"\n  TRUE TERMINATIONS (the number that drives the Phase B decision): "
      f"{N_TRUE_TERMS:,}")
print(f"  As % of all eligible ticker-months: "
      f"{100*N_TRUE_TERMS/max(len(elig),1):.4f}%")

print("\n--- True terminations BY YEAR ---")
if N_TRUE_TERMS:
    ty = TRUE_TERMS.groupby("year").size().reset_index(name="n_terminations")
    _elig_by_year = elig.groupby(elig["rebalance_date"].dt.year).size()
    ty["eligible_ticker_months"] = ty["year"].map(_elig_by_year)
    ty["pct"] = 100 * ty["n_terminations"] / ty["eligible_ticker_months"]
    print(f"  {'year':<6} {'terminations':>14} {'elig_tkr_mo':>13} {'pct':>8}")
    for _, r in ty.iterrows():
        mark = ""
        if int(r["year"]) == 2018:
            mark = "  <-- 2018: check H2 clustering (22% misclassification window)"
        print(f"  {int(r['year']):<6} {int(r['n_terminations']):>14,} "
              f"{int(r['eligible_ticker_months']):>13,} {r['pct']:>7.3f}%{mark}")

    # 2018 H1 vs H2 specifically.
    _t2018 = TRUE_TERMS[TRUE_TERMS["year"] == 2018]
    if len(_t2018):
        h1 = int((_t2018["rebalance_date"].dt.month <= 6).sum())
        h2 = int((_t2018["rebalance_date"].dt.month > 6).sum())
        print(f"\n  2018 split: H1={h1}  H2={h2}")
        print(f"    (H2 is the carried-forward membership window -- "
              f"clustering here is a DATA explanation, not a finding.)")
        RUN_LOG.append(("2018 H1/H2 terminations", "INFO", f"H1={h1} H2={h2}"))
else:
    print("  (none)")

print("\n--- True terminations BY PROVISIONAL DECILE  [THE DECISIVE CROSS-TAB] ---")
if N_TRUE_TERMS:
    td_ = TRUE_TERMS.groupby("prov_decile").size().reindex(range(1, 11), fill_value=0)
    _elig_by_dec = elig.groupby("prov_decile").size().reindex(range(1, 11), fill_value=0)
    print(f"  {'decile':<8} {'terminations':>14} {'eligible':>12} {'rate%':>10}")
    for d in range(1, 11):
        n_t = int(td_.loc[d])
        n_e = int(_elig_by_dec.loc[d])
        rate = 100 * n_t / n_e if n_e else float("nan")
        mark = ""
        if d in (1, 10):
            mark = "  <-- extreme decile (drives the long-short spread)"
        print(f"  {d:<8} {n_t:>14,} {n_e:>12,} {rate:>9.4f}%{mark}")

    d1, d10 = int(td_.loc[1]), int(td_.loc[10])
    mid = int(td_.loc[2:9].sum())
    print(f"\n  D1={d1}  D10={d10}  D2-D9 combined={mid}")
    print(f"  INTERPRETATION RULE (agreed in advance):")
    print(f"    - Few and evenly spread across deciles => the three bracketing")
    print(f"      assumptions converge and the exit choice is moot.")
    print(f"    - Clustered in D10 or D1, or in one period => the exit treatment")
    print(f"      needs resolving with corporate-action data BEFORE the spread")
    print(f"      means anything.")
    RUN_LOG.append(("true terminations D1/D10/mid", "INFO",
                    f"D1={d1} D10={d10} D2-D9={mid}"))
else:
    print("  (none)")

# -----------------------------------------------------------------------------
# Write per-termination detail to CSV. Aggregates go in the A7 summary; the
# full detail is too large to paste, so A7 prints this path instead.
# -----------------------------------------------------------------------------
os.makedirs(os.path.join(PROJECT_ROOT, "validation"), exist_ok=True)
_today = datetime.now().strftime("%Y-%m-%d")
TERMINATION_CSV = os.path.join(
    PROJECT_ROOT, "validation", f"momentum_termination_diagnostic_{_today}.csv")

terminations.to_csv(TERMINATION_CSV, index=False)
print(f"\n  Per-termination detail written to:")
print(f"    {TERMINATION_CSV}")
print(f"    ({len(terminations):,} rows, all classifications)")

print("\n" + "=" * 78)
print(f"CELL A5 COMPLETE: {N_TRUE_TERMS:,} true terminations identified")
print("=" * 78)

--- Provisional decile ranks (for BUCKETING terminations only) ---
  Eligible ticker-months: 85,184
  Rebalance months with <10 eligible names (no deciles): 0
  Decile distribution (should be near-uniform):
    decile  1:    8,446
    decile  2:    8,527
    decile  3:    8,499
    decile  4:    8,534
    decile  5:    8,538
    decile  6:    8,500
    decile  7:    8,511
    decile  8:    8,522
    decile  9:    8,504
    decile 10:    8,603

--- Detecting mid-month terminations ---
  Panel end: 2026-07-10
  Panel-edge cutoff: last obs after 2026-06-30 => classified 'panel_edge', not a termination

  Incomplete forward months detected: 42,144

--- Terminations by classification ---
    masked_gap_resumes           41,885  ( 99.4%)
    panel_edge                      222  (  0.5%)
    true_termination                 37  (  0.1%)

  TRUE TERMINATIONS (the number that drives the Phase B decision): 37
  As % of all eligible ticker-months: 0.0434%

--- True terminations BY YEAR ---
  year

## Cell A6 — Liquidity tercile construction

**Turnover window: `[t-21, t-1]` — the rebalance day's own turnover is excluded.**
You asked for no leakage of any kind, so although close-of-day turnover on `t` is
technically known at the moment of forming the portfolio, ending the window at
`t-1` makes it unambiguously strictly-past and removes the argument entirely.

Trailing mean rather than single-day because one unusual day misassigns a stock
to the wrong tercile, and a 21-day mean is closer to *"could I actually trade
this at size"*.

Two outputs matter for reading Phase B later:

- **Boundaries in absolute INR.** A relative tercile is meaningless if the bottom
  third still trades crores per day. If median T1 is a few lakh, the illiquid
  tercile is genuinely untradeable and any spread concentrated there is not a
  strategy.
- **Decile × tercile contingency table**, computed *before* any return is
  measured. Skew of extreme deciles toward the illiquid tercile is common and
  expected — flagged now so it is context rather than a surprise later.

In [7]:
# =============================================================================
# CELL A6 -- Liquidity tercile construction check
# =============================================================================
# PURPOSE: build the trailing-turnover terciles that Phase B's decomposition
# will use, and verify they are (a) economically meaningful and (b) not
# leaking.
#
# TURNOVER WINDOW (confirmed decision): trailing 21-row mean over [t-21, t-1].
#   The rebalance day's OWN turnover is EXCLUDED. You asked for no leakage of
#   any kind, so even though close-of-day turnover on t is technically known at
#   the moment of forming the portfolio, we end the window at t-1. This makes
#   the window unambiguously strictly-past and removes the argument entirely.
#
# WHY TRAILING MEAN RATHER THAN SINGLE-DAY:
#   Single-day turnover is very noisy -- one unusual day misassigns a stock to
#   the wrong tercile. A 21-day mean is more stable and closer to "could I
#   actually trade this at size".
#
# NOTHING IS WRITTEN. Read-only.
# =============================================================================

TURNOVER_WINDOW = 21   # trading rows
TURNOVER_MIN_VALID = 15  # require at least this many valid turnover obs

print("--- Trailing turnover specification ---")
print(f"  Window: [t-{TURNOVER_WINDOW}, t-1] inclusive = {TURNOVER_WINDOW} rows")
print(f"  Rebalance-day turnover EXCLUDED (strictly-past window, no leakage)")
print(f"  Minimum valid observations required: {TURNOVER_MIN_VALID}")

# Per-ticker turnover arrays, same positional convention as A2.
turnover_data = {}
for tkr, grp in panel.groupby(TICKER_COL, sort=False):
    turnover_data[tkr] = grp["turnover"].to_numpy(dtype=float)


def trailing_mean_turnover(ticker, as_of, window=TURNOVER_WINDOW):
    """Mean turnover over the `window` trading rows STRICTLY BEFORE `as_of`.

    `as_of` is explicit in the signature -- the point-in-time assumption is
    part of the interface, not a comment.

    Returns (mean_turnover, n_valid, window_start_date, window_end_date).
    """
    td = ticker_data.get(ticker)
    if td is None:
        return np.nan, 0, pd.NaT, pd.NaT

    # Same value-based lookup as A2 -- see _position_of() and the unit-hashing
    # note there. Must NOT revert to a dict lookup.
    p = _position_of(td["dates"], as_of)
    if p is None:
        return np.nan, 0, pd.NaT, pd.NaT

    start = p - window
    end = p - 1  # INCLUSIVE -- rebalance day itself excluded
    if start < 0:
        return np.nan, 0, pd.NaT, pd.NaT

    vals = turnover_data[ticker][start:end + 1]
    dates = td["dates"][start:end + 1]
    n_valid = int(np.sum(~np.isnan(vals)))
    if n_valid < TURNOVER_MIN_VALID:
        return np.nan, n_valid, pd.Timestamp(dates[0]), pd.Timestamp(dates[-1])
    return (float(np.nanmean(vals)), n_valid,
            pd.Timestamp(dates[0]), pd.Timestamp(dates[-1]))


print("\n--- Computing trailing turnover for eligible ticker-months ---")
_tov_vals, _tov_n, _tov_ws, _tov_we = [], [], [], []
for _, r in elig.iterrows():
    v, n, ws, we = trailing_mean_turnover(r[TICKER_COL], r["rebalance_date"])
    _tov_vals.append(v); _tov_n.append(n); _tov_ws.append(ws); _tov_we.append(we)

elig["trailing_turnover_21d"] = _tov_vals
elig["turnover_n_valid"] = _tov_n
elig["turnover_window_start"] = _tov_ws
elig["turnover_window_end"] = _tov_we

_n_tov_valid = int(elig["trailing_turnover_21d"].notna().sum())
print(f"  Valid trailing turnover: {_n_tov_valid:,} / {len(elig):,} "
      f"({100*_n_tov_valid/max(len(elig),1):.2f}%)")

# -----------------------------------------------------------------------------
# LEAKAGE ASSERTION on the turnover window.
# -----------------------------------------------------------------------------
_tw = elig[elig["turnover_window_end"].notna()]
_tw_violations = int((_tw["turnover_window_end"] >= _tw["rebalance_date"]).sum())
check("turnover window ends strictly BEFORE rebalance date (no leakage)",
      _tw_violations == 0, f"violations={_tw_violations:,}")

# -----------------------------------------------------------------------------
# Tercile assignment: cross-sectional within each rebalance date.
# -----------------------------------------------------------------------------
# Same pattern as the decile assignment in A5: assign in place rather than via
# groupby().apply(), which can consume the grouping column.
elig["liq_tercile"] = np.nan
for _rd, _idx in elig.groupby("rebalance_date").groups.items():
    _grp = elig.loc[_idx]
    _valid_idx = _grp.index[_grp["trailing_turnover_21d"].notna()]
    if len(_valid_idx) < 3:
        continue
    _r = elig.loc[_valid_idx, "trailing_turnover_21d"].rank(method="first", ascending=True)
    elig.loc[_valid_idx, "liq_tercile"] = np.ceil(3 * _r / len(_valid_idx)).clip(1, 3)

print("\n  Tercile counts (1 = LEAST liquid, 3 = MOST liquid):")
for t, c in elig["liq_tercile"].value_counts().sort_index().items():
    print(f"    tercile {int(t)}: {c:>8,}")

# -----------------------------------------------------------------------------
# ECONOMIC SANITY: what does "illiquid tercile" actually mean in rupees?
# A relative tercile is meaningless if the bottom third still trades crores
# per day. Boundaries must be readable in absolute terms.
# -----------------------------------------------------------------------------
print("\n--- Tercile boundaries by year (INR, trailing 21d mean turnover) ---")
_bnd = []
for yr, grp in elig[elig["trailing_turnover_21d"].notna()].groupby(
        elig["rebalance_date"].dt.year):
    q33 = grp["trailing_turnover_21d"].quantile(1 / 3)
    q67 = grp["trailing_turnover_21d"].quantile(2 / 3)
    med_t1 = grp.loc[grp["liq_tercile"] == 1, "trailing_turnover_21d"].median()
    med_t3 = grp.loc[grp["liq_tercile"] == 3, "trailing_turnover_21d"].median()
    _bnd.append({"year": yr, "q33": q33, "q67": q67,
                 "median_T1": med_t1, "median_T3": med_t3})
tercile_bounds = pd.DataFrame(_bnd)

print(f"  {'year':<6} {'T1/T2 bound':>16} {'T2/T3 bound':>16} "
      f"{'median T1':>16} {'median T3':>16}")
for _, r in tercile_bounds.iterrows():
    print(f"  {int(r['year']):<6} {r['q33']:>16,.0f} {r['q67']:>16,.0f} "
          f"{r['median_T1']:>16,.0f} {r['median_T3']:>16,.0f}")

print("\n  Read these in crores: divide by 1e7. If median T1 is only a few")
print("  lakh/day, the 'illiquid tercile' is genuinely untradeable at size and")
print("  any spread concentrated there is not a real strategy.")

# -----------------------------------------------------------------------------
# CONTINGENCY TABLE: momentum decile x liquidity tercile.
# Computed BEFORE any return is measured. If extreme deciles are strongly
# skewed toward the illiquid tercile, that is CONTEXT for reading Phase B's
# decomposition -- not itself a problem, but it should not be a surprise later.
# -----------------------------------------------------------------------------
print("\n--- Contingency: momentum decile x liquidity tercile (% within decile) ---")
_ct = elig[elig["prov_decile"].notna() & elig["liq_tercile"].notna()]
ct = pd.crosstab(_ct["prov_decile"], _ct["liq_tercile"], normalize="index") * 100
ct.columns = [f"T{int(c)}" for c in ct.columns]

print(f"  {'decile':<8} {'T1(illiq)%':>12} {'T2%':>10} {'T3(liq)%':>12}")
for d in ct.index:
    row = ct.loc[d]
    mark = "  <-- extreme decile" if int(d) in (1, 10) else ""
    print(f"  {int(d):<8} {row.get('T1', np.nan):>11.1f}% "
          f"{row.get('T2', np.nan):>9.1f}% {row.get('T3', np.nan):>11.1f}%{mark}")

print("\n  Uniform would be 33.3/33.3/33.3. Skew in D1/D10 toward T1 is common")
print("  and expected (extreme momentum is more prevalent in small caps); it is")
print("  flagged here so Phase B's tercile decomposition is read in context.")

print("\n" + "=" * 78)
print("CELL A6 COMPLETE: terciles built, leakage-checked, economically sized")
print("=" * 78)

--- Trailing turnover specification ---
  Window: [t-21, t-1] inclusive = 21 rows
  Rebalance-day turnover EXCLUDED (strictly-past window, no leakage)
  Minimum valid observations required: 15

--- Computing trailing turnover for eligible ticker-months ---
  Valid trailing turnover: 85,170 / 85,184 (99.98%)
  [PASS] turnover window ends strictly BEFORE rebalance date (no leakage)  --  violations=0

  Tercile counts (1 = LEAST liquid, 3 = MOST liquid):
    tercile 1:   28,328
    tercile 2:   28,392
    tercile 3:   28,450

--- Tercile boundaries by year (INR, trailing 21d mean turnover) ---
  year        T1/T2 bound      T2/T3 bound        median T1        median T3
  2011         10,069,857       86,051,551        3,040,692      283,255,073
  2012         13,210,320       98,359,417        4,398,272      317,534,040
  2013          9,475,943       74,831,899        3,352,048      331,332,642
  2014         28,006,254      140,434,618       11,790,945      511,483,914
  2015         27

## Cell A7 — Paste-back summary

Emits one fenced block containing everything needed for the Phase B decisions.
Copy the whole thing between the markers.

**This cell re-reads variables produced by A0–A6. It recomputes nothing** — if it
recomputed, it could disagree with what you just read above it, and we would end
up debugging the summary instead of the test.

Aggregates and the decisive cross-tabs are inlined; full per-termination detail
stays in the CSV, whose path is printed in case a specific slice is needed. The
block ends with a **"decisions this output should drive"** section that attaches
each open question to the specific numbers bearing on it.

In [8]:
# =============================================================================
# CELL A7 -- SINGLE PASTE-BACK SUMMARY BLOCK
# =============================================================================
# Emits one fenced text block containing everything needed to make the Phase B
# decisions. Copy the ENTIRE block between the BEGIN/END markers and paste it
# back in one shot.
#
# HARD CONSTRAINT ON THIS CELL: it RE-READS variables produced by A0-A6. It
# does NOT recompute anything. If it recomputed, it could disagree with what
# you just read above it, and we would end up debugging the summary instead of
# the test.
#
# Full per-termination detail is NOT inlined (too large to paste) -- the CSV
# path is printed instead so a specific slice can be requested if needed.
# =============================================================================

L = []
A = L.append

A("=" * 78)
A("MOONSHOT_COMPOUNDING -- TEST 1 MOMENTUM -- PHASE A DIAGNOSTIC SUMMARY")
A("=" * 78)

# --- Run header --------------------------------------------------------------
A("")
A("[1] RUN HEADER")
A(f"  run_timestamp     : {RUN_TIMESTAMP}")
A(f"  panel_file        : {PANEL_FILENAME}")
A(f"  panel_mtime       : {PANEL_MTIME}")
A(f"  panel_rows        : {PANEL_ROWS:,}   (v3: v2 minus 4,747 holiday rows)")
A(f"  panel_dates       : {PANEL_DATES:,}")
A(f"  panel_tickers     : {PANEL_TICKERS:,}")
A(f"  panel_date_range  : {PANEL_START.date()} -> {PANEL_END.date()}")
A(f"  ticker_col        : {TICKER_COL}")
A(f"  date_col          : {DATE_COL}")
A(f"  rebalance_dates   : {N_REBALANCES}  "
  f"({REBALANCE_DATES[0].date()} -> {REBALANCE_DATES[-1].date()})")
A(f"  signal_spec       : lookback={LOOKBACK} skip={SKIP} "
  f"window_rows={EXPECTED_WINDOW_ROWS} strict_min={STRICT_MIN}")
A(f"  turnover_spec     : trailing mean over [t-{TURNOVER_WINDOW}, t-1], "
  f"min_valid={TURNOVER_MIN_VALID}")

# --- Check ledger ------------------------------------------------------------
A("")
A("[2] CHECK LEDGER (every assert run in A0-A6)")
_n_pass = sum(1 for _, s, _ in RUN_LOG if s == "PASS")
_n_fail = sum(1 for _, s, _ in RUN_LOG if s == "FAIL")
_n_info = sum(1 for _, s, _ in RUN_LOG if s in ("INFO", "SKIP"))
A(f"  PASS={_n_pass}  FAIL={_n_fail}  INFO/SKIP={_n_info}")
A("")
for nm, st, dt in RUN_LOG:
    A(f"  [{st:<4}] {nm}" + (f"  --  {dt}" if dt else ""))

# --- Coverage ----------------------------------------------------------------
A("")
A("[3] SIGNAL COVERAGE -- strict(232) vs relaxed(186) diagnostic")
A(f"  OVERALL strict  : {_overall_strict:.2f}%")
A(f"  OVERALL relaxed : {_overall_relaxed:.2f}%")
A(f"  Relaxing 232->186 would add "
  f"{int(signals['would_pass_relaxed'].sum() - signals['has_strict'].sum()):,} "
  f"ticker-months ({_overall_relaxed - _overall_strict:+.2f}pp)")
A("")
A(f"  {'year':<6} {'ticker_mo':>10} {'strict':>9} {'strict%':>8} "
  f"{'relaxed':>9} {'relax%':>8} {'extra':>8}")
for _, r in coverage_by_year.iterrows():
    A(f"  {int(r['year']):<6} {int(r['ticker_months']):>10,} "
      f"{int(r['strict_valid']):>9,} {r['strict_pct']:>7.1f}% "
      f"{int(r['relaxed_valid']):>9,} {r['relaxed_pct']:>7.1f}% "
      f"{int(r['extra_from_relaxing']):>8,}")

A("")
A("  NaN attribution (why a signal is missing):")
for reason, cnt in reason_counts.items():
    A(f"    {reason:<34} {cnt:>10,}  ({100*cnt/len(signals):>5.2f}%)")

A("")
A("  Strict-filter drop rate among names WITH a full 232-row window:")
A(f"  {'year':<6} {'avg_avail':>11} {'avg_passed':>12} {'avg_dropped':>13} {'avg_drop%':>11}")
for _, r in drop_by_year.iterrows():
    A(f"  {int(r['year']):<6} {r['mean_had_window']:>11.1f} "
      f"{r['mean_passed']:>12.1f} {r['mean_dropped']:>13.1f} "
      f"{r['mean_drop_pct']:>10.1f}%")

# --- Leakage / spot-check ----------------------------------------------------
A("")
A("[4] LEAKAGE CHECK + MANUAL SPOT-CHECK")
A(f"  {'ticker':<16} {'rebalance':<12} {'win_start':<12} {'win_end':<12} "
  f"{'rows':>5} {'skip':>5} {'manual':>13} {'vector':>13} {'diff':>10} {'ok':>4}")
for _, r in spot_check_df.iterrows():
    A(f"  {str(r['ticker']):<16} {str(r['rebalance_date'].date()):<12} "
      f"{str(r['window_start'].date()):<12} {str(r['window_end'].date()):<12} "
      f"{int(r['n_rows']):>5} {int(r['rows_skipped']):>5} "
      f"{r['manual_signal']:>13.8f} {r['vectorised_signal']:>13.8f} "
      f"{r['abs_diff']:>10.2e} {'YES' if r['agree'] else 'NO':>4}")
A("")
A(f"  GLOBAL: signals with window_end >= rebalance_date : {len(_violations):,} "
  f"(of {len(_ok):,}) -- must be 0")
A(f"  GLOBAL calendar gap window_end->rebalance: min={_gaps.min()} "
  f"p01={_gaps.quantile(0.01):.0f} median={_gaps.median():.0f} "
  f"p99={_gaps.quantile(0.99):.0f} max={_gaps.max()}")

# --- Universe ----------------------------------------------------------------
A("")
A("[5] PIT UNIVERSE CHECK (filter: in_nifty500_flag == 1)")
A("  Flag state distribution across all (ticker, rebalance) pairs:")
for st, cnt in signals["flag_state"].value_counts().items():
    A(f"    {st:<12} {cnt:>10,}  ({100*cnt/len(signals):>5.2f}%)")
A("")
A(f"  {'year':<6} {'flagged(avg)':>13} {'elig(avg)':>11} {'elig(min)':>11} "
  f"{'elig(max)':>11} {'retained%':>11}")
for _, r in univ_by_year.iterrows():
    A(f"  {int(r['year']):<6} {r['mean_flagged']:>13.1f} {r['mean_eligible']:>11.1f} "
      f"{int(r['min_eligible']):>11,} {int(r['max_eligible']):>11,} "
      f"{r['pct_retained']:>10.1f}%")
A("")
A(f"  Eligible universe variation: mean={_elig_mean:.1f} std={_elig_std:.1f} "
  f"(near-zero std would indicate the PIT filter is not being applied)")
A("")
A("  Named ground-truth checks (fact #18: semi-annual grid, ~6mo slack expected):")
for nm, st, dt in RUN_LOG:
    if nm.startswith("named check"):
        A(f"    [{st}] {nm}: {dt}")

# --- Terminations ------------------------------------------------------------
A("")
A("[6] TERMINATION DIAGNOSTIC  [DECIDES PHASE B EXIT TREATMENT]")
A(f"  Total incomplete forward months : {len(terminations):,}")
A("  By classification:")
if len(terminations):
    for k, c in terminations["classification"].value_counts().items():
        A(f"    {k:<26} {c:>8,}  ({100*c/len(terminations):>5.1f}%)")
else:
    A("    (none detected)")
A("")
A(f"  TRUE TERMINATIONS: {N_TRUE_TERMS:,}  "
  f"({100*N_TRUE_TERMS/max(len(elig),1):.4f}% of eligible ticker-months)")

if N_TRUE_TERMS:
    A("")
    A("  By year:")
    A(f"  {'year':<6} {'terminations':>14} {'elig_tkr_mo':>13} {'pct':>8}")
    for _, r in ty.iterrows():
        A(f"  {int(r['year']):<6} {int(r['n_terminations']):>14,} "
          f"{int(r['eligible_ticker_months']):>13,} {r['pct']:>7.3f}%")
    _t2018 = TRUE_TERMS[TRUE_TERMS["year"] == 2018]
    if len(_t2018):
        A(f"  2018 split: H1={int((_t2018['rebalance_date'].dt.month <= 6).sum())} "
          f"H2={int((_t2018['rebalance_date'].dt.month > 6).sum())}")
    A("")
    A("  By provisional decile  [DECISIVE CROSS-TAB]:")
    A(f"  {'decile':<8} {'terminations':>14} {'eligible':>12} {'rate%':>10}")
    for d in range(1, 11):
        n_t = int(td_.loc[d]); n_e = int(_elig_by_dec.loc[d])
        A(f"  {d:<8} {n_t:>14,} {n_e:>12,} "
          f"{(100*n_t/n_e if n_e else float('nan')):>9.4f}%")
    A(f"  D1={d1}  D10={d10}  D2-D9 combined={mid}")

A("")
A(f"  Full per-termination detail CSV: {TERMINATION_CSV}")

# --- Terciles ----------------------------------------------------------------
A("")
A("[7] LIQUIDITY TERCILE CONSTRUCTION")
A(f"  Valid trailing turnover: {_n_tov_valid:,} / {len(elig):,} "
  f"({100*_n_tov_valid/max(len(elig),1):.2f}%)")
A(f"  Turnover-window leakage violations: {_tw_violations:,} (must be 0)")
A("")
A(f"  Tercile boundaries by year (INR; divide by 1e7 for crores):")
A(f"  {'year':<6} {'T1/T2 bound':>16} {'T2/T3 bound':>16} "
  f"{'median T1':>16} {'median T3':>16}")
for _, r in tercile_bounds.iterrows():
    A(f"  {int(r['year']):<6} {r['q33']:>16,.0f} {r['q67']:>16,.0f} "
      f"{r['median_T1']:>16,.0f} {r['median_T3']:>16,.0f}")
A("")
A("  Momentum decile x liquidity tercile (% within decile; uniform = 33.3 each):")
A(f"  {'decile':<8} {'T1(illiq)%':>12} {'T2%':>10} {'T3(liq)%':>12}")
for d in ct.index:
    row = ct.loc[d]
    A(f"  {int(d):<8} {row.get('T1', float('nan')):>11.1f}% "
      f"{row.get('T2', float('nan')):>9.1f}% {row.get('T3', float('nan')):>11.1f}%")

# --- Failures repeated -------------------------------------------------------
A("")
A("[8] FAILED CHECKS (repeated at bottom so they cannot be missed)")
if FAILED_CHECKS:
    for nm, dt in FAILED_CHECKS:
        A(f"  [FAIL] {nm}  --  {dt}")
else:
    A("  None. All checks passed.")

# --- Decisions this output should drive --------------------------------------
A("")
A("[9] DECISIONS THIS OUTPUT SHOULD DRIVE")
A("")
A("  Q1. EXIT TREATMENT for mid-month terminations.")
A(f"      Evidence: {N_TRUE_TERMS:,} true terminations "
  f"({100*N_TRUE_TERMS/max(len(elig),1):.4f}% of eligible ticker-months).")
if N_TRUE_TERMS:
    A(f"      Decile concentration: D1={d1}, D10={d10}, D2-D9={mid}.")
    A(f"      2018 H2 count: "
      f"{int((TRUE_TERMS[TRUE_TERMS['year']==2018]['rebalance_date'].dt.month > 6).sum())}")
A("      If few and evenly spread -> the three bracketing assumptions")
A("      (observed-days / -100% / flat-0%) converge; choice is moot.")
A("      If clustered in D10 or D1 -> exit treatment drives the spread and")
A("      must be resolved with corporate-action data before Phase B runs.")
A("")
A("  Q2. IS THE 232 FILTER DOING MORE WORK THAN INTENDED?")
A(f"      Evidence: strict {_overall_strict:.2f}% vs relaxed {_overall_relaxed:.2f}% "
  f"({_overall_relaxed - _overall_strict:+.2f}pp).")
A(f"      Mean names dropped per month (among those with a full window): "
  f"{drop_by_month['dropped'].mean():.1f}")
A("      If strict and relaxed are close -> your hard-232 decision was free.")
A("      If they diverge materially -> the filter is excluding names for")
A("      data-artifact reasons (fact #14 phantom masking), not tradeability.")
A("")
A("  Q3. ANY FAILED CHECK ABOVE -> resolve before Phase B is written.")
A("")
A("  Q4. DID THE v3 FIX RESTORE THE COLLAPSED YEARS?")
A("      Pre-fix eligible universe: 2025 = 125.7, 2026 = 22.3 (vs ~480 normal).")
A("      Target after fix: ~450-490 in both.")
for _y in (2014, 2015, 2024, 2025, 2026):
    _r = univ_by_year[univ_by_year["year"] == _y]
    if len(_r):
        A(f"      {_y}: eligible avg = {_r.iloc[0]['mean_eligible']:.1f}  "
          f"(retained {_r.iloc[0]['pct_retained']:.1f}%)")
A("      If any remain depressed, the six deleted dates were NOT the whole story.")

A("")
A("=" * 78)
A("END PHASE A SUMMARY")
A("=" * 78)

SUMMARY_TEXT = "\n".join(L)

print("\n\n")
print(">>> COPY EVERYTHING BETWEEN THE MARKERS BELOW AND PASTE BACK <<<")
print("\n----- BEGIN PHASE A SUMMARY -----")
print(SUMMARY_TEXT)
print("----- END PHASE A SUMMARY -----")

# Also save to disk in case the Colab output cell is truncated.
_summary_path = os.path.join(
    PROJECT_ROOT, "validation",
    f"momentum_phaseA_summary_{datetime.now().strftime('%Y-%m-%d')}.txt")
with open(_summary_path, "w") as f:
    f.write(SUMMARY_TEXT)
print(f"\n(Also saved to: {_summary_path})")




>>> COPY EVERYTHING BETWEEN THE MARKERS BELOW AND PASTE BACK <<<

----- BEGIN PHASE A SUMMARY -----
MOONSHOT_COMPOUNDING -- TEST 1 MOMENTUM -- PHASE A DIAGNOSTIC SUMMARY

[1] RUN HEADER
  run_timestamp     : 2026-07-24 14:08:17
  panel_file        : feature_panel_with_pit_v2_2026-07-24.parquet
  panel_mtime       : 2026-07-24 07:52:10
  panel_rows        : 3,328,132   (v3: v2 minus 4,747 holiday rows)
  panel_dates       : 4,237
  panel_tickers     : 1,069
  panel_date_range  : 2010-01-04 -> 2026-07-10
  ticker_col        : ticker
  date_col          : Date
  rebalance_dates   : 199  (2010-01-29 -> 2026-07-10)
  signal_spec       : lookback=252 skip=21 window_rows=232 strict_min=232
  turnover_spec     : trailing mean over [t-21, t-1], min_valid=15

[2] CHECK LEDGER (every assert run in A0-A6)
  PASS=41  FAIL=1  INFO/SKIP=9

  [PASS] project root directory exists  --  /content/drive/MyDrive/moonshot_compounding
  [PASS] parquet files visible under project root  --  1592 files found


In [9]:
# =============================================================================
# 2024 RESIDUAL DIAGNOSTIC — why is retention 81.8% vs ~95% elsewhere?
# =============================================================================
import numpy as np

short = signals[signals["signal_reason"] == "insufficient_valid_returns"].copy()
short["missing"] = short["n_required"] - short["n_valid_counted"]
short["yr"] = short["rebalance_date"].dt.year

print("--- [1] Missing-count distribution by year ---")
print("If 2024's median is ~2, it's a date-driven event (like the holidays).")
print("If it's large and spread, it's genuine per-stock gaps.\n")
print(short.groupby("yr")["missing"].describe(percentiles=[.25,.5,.75,.9]).round(1).to_string())

print("\n--- [2] Distinct tickers failing, by year ---")
print("Few tickers = concentrated problem. Hundreds = universe-wide event.\n")
print(short.groupby("yr")[TICKER_COL].nunique().to_string())

print("\n--- [3] Which 2024 rebalance months are worst? ---")
s24 = short[short["yr"] == 2024]
by_month = s24.groupby(s24["rebalance_date"].dt.to_period("M")).agg(
    n_failing=(TICKER_COL, "size"),
    n_tickers=(TICKER_COL, "nunique"),
    median_missing=("missing", "median"))
print(by_month.to_string())

print("\n--- [4] Top failing tickers in 2024 ---")
print("Recognisable large-caps = data artifact. Obscure/dead names = real illiquidity.\n")
print(s24[TICKER_COL].value_counts().head(15).to_string())

print("\n--- [5] Where are the NaN returns concentrated? ---")
print("Any single date with a large NaN count is the smoking gun.\n")
w = panel[(panel["Date"] >= "2023-01-01") & (panel["Date"] <= "2024-12-31")]
nan_by_date = w[w["daily_return"].isna()].groupby("Date").size().sort_values(ascending=False)
tot_by_date = w.groupby("Date").size()
top = nan_by_date.head(15).index
out = pd.DataFrame({
    "nan_returns": nan_by_date.loc[top],
    "total_rows": tot_by_date.loc[top],
    "pct_nan": (100 * nan_by_date.loc[top] / tot_by_date.loc[top]).round(1),
    "masked": [int((w[w["Date"]==d]["price_status"] != "observed").sum()) for d in top],
    "real_vol_rows": [int((w[w["Date"]==d]["Volume"].fillna(0) > 0).sum()) for d in top],
}).sort_index()
print(out.to_string())

--- [1] Missing-count distribution by year ---
If 2024's median is ~2, it's a date-driven event (like the holidays).
If it's large and spread, it's genuine per-stock gaps.

            count       mean        std      min       25%        50%        75%        90%        max
yr                                                                                                    
2011   727.000000  45.500000  76.900000 1.000000  2.000000  14.000000  23.000000 232.000000 232.000000
2012 1,082.000000  32.700000  63.800000 2.000000  2.000000   9.000000  23.000000  72.800000 232.000000
2013   808.000000  51.500000  76.400000 1.000000  6.000000  21.000000  35.000000 232.000000 232.000000
2014   883.000000  46.000000  66.900000 1.000000  2.000000  22.000000  46.000000 155.800000 232.000000
2015   779.000000  47.300000  79.000000 2.000000  2.000000   8.000000  28.000000 232.000000 232.000000
2016   689.000000  59.400000  88.700000 1.000000  2.000000  10.000000  67.000000 232.000000 232.000000
201

---

## After this notebook

1. Paste the A7 block back.
2. Decide the **exit treatment** from the termination counts and their decile
   concentration.
3. Decide whether the **232 filter** stays hard, based on the strict-vs-186 gap.
4. Resolve any **failed check** — Phase B is not written until section [8] is empty.

Phase B will then be built as `moonshot_signals.py` (shared functions, imported by
both notebooks so the leakage proof provably tested the same code the test runs)
plus the re-runnable test notebook. Writing the signal function as
`momentum_signal(panel, as_of, lookback=252, skip=21)` makes reversal a parameter
change rather than a copy-paste — reversal is already among the six planned tests,
so this is not speculative generalization.